# Deney 06 — Render Pipeline (Notebook 01+02+03 çıktılarını birleştirir)

**Amaç:** Onaylanmış an + script + seslendirmeyi kullanarak dikey (9:16) bir YouTube Shorts videosu üretmek: kaynak aralığı kes, dikey kadraj uygula, seslendirmeyi orijinal sesle miksle (ducking), altyazı yak, kaliteli kodlama ile dışa aktar.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → Notebook 03 → seslendirme onayı → **bu notebook**. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Bu notebook hiçbir yere yükleme/yayınlama yapmaz.** Çıktı sadece `results/render/final_short.mp4` olarak yerelde/Colab oturumunda kalır.

**Tek kaynak politikası (düzeltildi):** Bu notebook artık ffmpeg filtergraph'larını, altyazı üretimini, doğrulamayı ve onay kapısı mantığını **kendi içinde tekrar tanımlamaz**. Hepsi `local_pipeline/render_utils.py` içindeki kanonik fonksiyonlardan gelir — master notebook (`00_...`) ve `verify_local.py` de **aynı** fonksiyonları çağırır, yani üç ayrı kopya yoktur. Modül aşağıdaki bootstrap hücresiyle yüklenir (repo → Drive → `PROJECT_REPO_URL` → gömülü hash-doğrulamalı yedek).

**Onay politikası (üretimden sonra, içeriğe bağlı):** `run_render_pipeline()` **onay argümanı almaz** ve asla `approved` üretemez — yeni render her zaman `review_required` (teknik doğrulama başarısızsa `failed`) olur. Onay, videoyu **izledikten sonra** bölüm 5'teki ayrı hücrede, dosyanın SHA-256'sı doğrulanarak verilir. Yeniden render alırsan hash değişir ve önceki onay otomatik geçersizleşir. `render_validation` bir **teknik** kontrol sonucudur, insan onayı değildir.

In [ ]:
!pip -q install imageio-ffmpeg faster-whisper pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, json
from pathlib import Path

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RENDER_DIR = RESULTS_DIR / 'render'
RENDER_DIR.mkdir(parents=True, exist_ok=True)

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}
print('Sonuc dizini:', RESULTS_DIR)


## `local_pipeline` kanonik modüllerini yükle (tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['render_utils', 'video_analysis', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'render_utils': {
        'sha256': '53d5053906b6586f80fbd0890c7680173e054e2fc6b45aa2b6210ef250a06833',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK+m_q7mFRnY1-g38P-?SO7hl?uqt0^M)oGu&?OwJ!Jz3>CG9^(ZMii;=Qf0YDYt0{=m-(FYHS+`K7tX!_AOM1@@+Fg*Oq{e$'
            'k-)}nV`Ja3zrX(~uFGhW#^Q$`R_pje6md3-i-UEtj?*NI#Uf7EaZ!qL5f$-FOg7@^!I%q;RiQz$-eeQeyILgE#a@!(yHZ4%i2qV2'
            'mr)vLRVbe1X*4-VpyC>9XO)=d>y60gqFN-SNb`$i+J{OraTyoMd^1k-X_OAun>~grlGQpds*;PZ@>va?o+U**t<sH{=LLL-1yz!R'
            '5alXg#-fVLsz2D<dyzxU>;gtq#G+oOc{D2pR9n}RG$|L<;|&!Vh-a0^GXPt}Scrknla$KL#8r_1&^_~YO$|l@-lQ;nWxZaf3Cves'
            'SL?bG>!?}`_V)Mp_vS^u665i_uIeHlkL9QUY?fD1mE>8uw<q5%qH>WY6ZP}YWuB?$B34i3qOOuu|EeeJBA>=(soqoz90LrBTb@N#'
            'TqUbmwX1%I0{{MyXEB$<4ny0j*w65DZ}0V+$8UalJ^th4=f9l379(-JC*aS1{2nj_YG+lH?uTN37GK6`zQz~hbsA0MMV`X!;HM(K'
            'OyVoRA3zL1r~Qy2L~H2vCH@xsNmkbLc`{9~I&?3>m-7f_nLb0oSv|#x8soq=0AdS0`{nyL&)z&g9sl~{v)4bLz8e4Z^6BYw>SItP'
            '*(L}@P_AHY@L?6tl6r+tX;fUq2bXsbU@<0n0Dzu6{`t+bmoLU|emwo@bo}$Hmp?pvZj3dU#q+36t6(T!&`anOV!;>?w}g(t$K$a1'
            'I?rJsS+t7beeg8?bM(hrypFOmaLbg*2Q2gLiSZ$Vp<c%0X`bfAI4Vno@y|aV!Ji-Cj|pNDm7a1x{NR>bMeoN~c`?Hx_nh}r*yQ;B'
            '?veGOxJa_`CA>d5F+Tu~$Hf$yg)cuuX&D>e(zvRCLX_)h3b+RaWW_E26I{T2jPv;z&=YX3ylFm9y5{qzsF=oST9@MWlg<e}`kPPa'
            '$-|GG&^JZBdp-vy$f^K@nik2LHU`Y!YK=JB(<P|X(<ol$nRo;3N5v-2Lh))5#o4Ba%y)lG%DC8+=BuAJWmF^)>=+y1k9l3jX2D;9'
            '*2YC%hT>@vT>vqwV0A<B2k7(LpcK!6dctE1&a(o$MW_JnN9hI{!=A*jK{tQ^x^&-NV#JdcF1&tn@G8Ejq4J=r=54fCL?`zjV2z`@'
            'NB8bUlLz-tP99E<?%%&R{Y`X#dK^EvJDZ>UW_B;W|1kRY_~HF;<CEi)heyXzH2?O&!`XaxeDq*4n;*46*F}nO7ge<`hlhtnbTzn0'
            'szp74c`Bw*0*KmRny(Ho^86x=53y2txQdeOFrTOJAV*M64{tG=G?~U3AgMVCFMoJGsNPq;VqdUxo>nGs1Okl)s+>l&#qbEh_F(hG'
            '07pTSKn`v<L<Loi;Ul2ieUyafYw?G84aY6BC-32qo+ikNtk!_waKIiGd74h5=~Cphmj>c-UF9oib1HuS@{K6qKu0Db%daFnn7oo$'
            'k5vwQ1suRtQl;?$9!|tpnG{M$$|j--9CrgRDICKF{}892e8N28>K(N3!Nb2jE&TRN)57Dy$u7FUGji}pBw?^3H_(Lnz1hRNv%B|Z'
            '$MXl%Z|@x+|K@IdFPhHpO@JuOe{=utaeO~|@a^P56cOE+9UVpA&L-cUMEByO`Q3Y8nkFzh9P%Lk>Hhau7mMnj?mzw8(gNA3PbLG_'
            'pgLFP<2JvyPg`OSfBzVmxtBixSNH1Wi__PVyZgiGpMHJ$>M3$}y=E)2M8aGN98?QB;^-ZS8+54Q>q?z@_~S~Rda}(R!Q+f!;OAL$'
            '6w*&Rh4HD7r!T=sV9twqj4bM2e{XLOI89*l%RI#$8u7sneOSdk<ess`8UG%NvMOK<K?IZvK+)>Gug3V7{ucDb!FM1oO{*cZ+$i}G'
            'a1dH%F*LnOvZO>ZCmYkWisS)JDyc~nn-r3uP=gS4ub??-)a@l!;U4w}@Htuc`hzsTii=*KswZ;;!dGfELyZj9caW7tk_kVvKD2m?'
            'A_8^>d5StdEsDIbSt9&1-~T=ViFFmjYU;tunZ4F+epPHAiG5GyReU)`36EwAWIw1>g2)uldj9V9w>Ff%{v}s?7NXb*XAw-KZXPg1'
            'EdkFb^F%Rl-OYHjyZ{*o2>om<kwC76i4wFBfj#;72hO<Zf>tksQuZ|UmhlG3I*Hlp85ZD%eFbY~6mIv<;O6kp3uOQH+VcWnWapfv'
            'ZO>Ort_a0EkbM|lODNlhpS0(%3#bzMU=gyMr++MgjtOQUcL0cynR#qGdjLpQv!G@vZqg_u`{0zRmDpPn3&0Oyi%Wn{xY0ZC?@U2I'
            'm(#zM#73Thbb^fMb(;2&%o5!I9O7A;B^S_pm=md+7;qhlMptlx0N)g6R1*o@1n3U%T8jt?c{9<7M$<)|EfFsgAU?e`T1{rrP?j7Z'
            '_tblE|Nh+v{ZLE*UF}8UZVc)*sxNv3sc$ZU{Jx0a^HWa|R5(UZ?PAf>F&-a1euTM!h35GKN*V#WjAu#FCZ0b<OQf8dMgj;CT}DYt'
            'JRfO>(v6|bf>D-Yon#s8u-896K7dIZ>=Cf<`H&NQmcbkX&-pA9C9pQYMiO*VouSGrrc)?>4~!ceNvI@=Qt^8_p$XOxco33CN+hM2'
            '<yRSMxlj$2P2p8m<pkhq0f*=Ta=GwH>dI)W!0j^So5|UvPpEuBC0IQ}jTo$KjJmc6`-rMNcv-^APf583Ex>t_%;#}|Dn6W-ftoY='
            'SIyW+m@|H+arc}`DJ}+z$q$2zxav8)ojy76JWy|k20i8d$h9QTKtzT)vl5rKc~#~$aO%iblpENgE5ZxZ2_Y<v3eY1|hV>fUu)hyh'
            'OZXc$0FJ*L5$g&o21n~SU(!#mFaxwqGSc_~$OinO9taco_v?6y1cWzQGaqLb-+Jzzr3UyxQqq+5Si9QZ=0;h4EQry_ZTp6a#dF4l'
            '<V-RhLwe4v>82f$;IV+;)r`Y9{|wEKi#)H?q>s7R<NiPrF<>HirsY3WQ?ZTMF`lpn&|x!weopmV&7cY8#wj!l-);_BZ#zI8_=#0L'
            'G}o><1RY+ub6mVnc5}`lG63fB>J)E=Ym;4XpJkoU!eOg3Ezu-wM3a8g4@L+)Zxv{b7(^R=9FJ5_gEWJOkvv@Y^k>+`@E!PccZ&ss'
            'Yt@_WK+(yDEvSl(J#sqP>{bTwlg~P9L!w}H&f4_6u15Fn(*X$Rx&{%z;Wex-4#=u7o@}bP?5Xm65E_#t7p#qe8cu_LU+q9Yn>~J2'
            'XV_t8kb@+HS;E=jV@)1caGYEeHJsB?nTSb*3c)h@5ZGO7MpMM3`tNy>**nL#o@Y~rHKzLH=!U5#PWBVA^!BS0zn`Lp>6HFK>JEKU'
            'u;E*~Tk8T2=H5K`<xh`aJsa|zr%4%QrC7pF!|7Hlf|d)AFAQ*7X91c`r1*oeLT|mD4EER4*H#|^-I`^kldN<JgT3@M8UUBQ{@x9U'
            '<`Jge#H)yApaxFxLsFy>EJZX+MDN<(X0$@*+dj+&&ceVWn(hA(70WasaFHnE63|s##8cpmHqkms5$?c|#gwo+-P`NjY2M><*Ix4s'
            'eI6JiMJYpVvL(DJfYV0{DRx#ZxzVUv@-KS|^TXCrJj~~46dq3iT#>|j-AQ|Zt(T(L6aiUl4y!bY@kf7P{sIARBPKDiTAE?YVU*%O'
            '{_Wp+s#noPoaC}aw!W5&WKD`%`I5S^3Ar`{5251%5f@rO4m2hbvd7Y}U`9?=PqMNCw%qC$Aj33Yt*ANd+(0}b3vMY^H3&L&87pQ4'
            'D&Sf~u-MWdJ%^H|AZ0PIy}*DV&o17{)((<)eT!R$L#$q*A(l|x?gQK4wJt>-YxbP%$%<;aQMTs;nYshaF?|Aw3#2Q%DpEMLmht;o'
            'ZcVcfZCLcpMzxNmR<czIVij{3z&aPWV^Fed3z0C&Qz{FqM%Hs)C1Tf5GK^P~c!qW&T(Ja%o(2OL=0y>$7YI+yW)EoiK<nPi(Qgsf'
            'cYuOV@(ie6#j8}N5t;d6v7ni&sF(@$3b{J&e@Or%@hi~W{0f99l9gEHEKmcUiQbbTj_1kSx5K_(GDF+KOrSadgST(-C^>AT6*ymd'
            'r2Y`!iDPOSf1MqlBf}UB4W*X^N}rF;#T@~_h5`IbZ%&5i=IY=(xfGK+Nr~Q92?(NNW7c{oMG_qdVW-Th#Sq29P%Pr)Vo}K-le8|z'
            'Wpc5K=mqou&sJv%6#?)s&b){&&;da7U>m0FfpAy>B}{Q$Ck4u$uq>=;Bak_C7kB4$5drnUErV_j0A~g3b9OYmJZAwm5^7xH*Y`gv'
            '2MZ98W0a!LsKV@=g-#x90H-Ay45mfC4oLll11x}2BUW4#NQVMjN=20uircs#X6=>?WFKPled;J6QW}r0X@0lEYo6Qf5Rh&fkMknA'
            '06LZeQ@bYi9SQX)$)>2ziNk#dAU4|qrAE2$Xv@rQd&YRn-mlXHxT&+r#kee{=Vx>N_mSH(Rp75a+VBF$G9rW=T{p<Ng&ugiHTQMv'
            ';ACfqW-)gNDtiZtvv7_Wnns&Z@7IH?{^7~6_v66={_R?fmgafBKWP-rQn$hwdX4{lRcu9X5-))~#Mw|7XvaI!Ud;Y}-`ea^JwtMD'
            '?DR!c8l%O}Lovzo6a>tAx`b^w20j-~otb>?<(C_k8`R;+0u~k6NhA|!6ubmx1vTaJ9URb9D6>C&D)m?;?~z^rUPkAlr$!^DX|fhH'
            'B<{8J)gq<S6sQ2~gZAkAEM1%(6D-})K9iNR2q?+<z#ss~Wq9V$-9vyM2$6?7x^^dE%YZdhckXYoB^%(zj3e^HlLuRO_XghhtH}?M'
            '+Wr~xGqd;{A7B~0+X5@-g7_B^Z?vWZ16hvH_pC(cuhG%q=>Cwm+I53T+hJ5yfYBrP2Gkn{H@bUtbcO|BKeZc_A5@Pb_zOr0kpFUY'
            'GMv>s(h(fdsPi4pVCTTpkI6SCp-(3O-B=D|uey`}Q4{|OlWt8P&HLvLM%xVQOtN|gYc5axsJJMHC|p)&@Ec_3Z$e~kFKD%F^NZp@'
            '@(M*r*`0A2Pvwpl|H{n#i2mXIt(^}CPXGmgXNqS(^NVKR*Rbj3j;HN-un`9fz#JfLSq230AVqEVWt@r_i6-zm*y(|Ih19gPq^QQ`'
            'fJ$xD>DTC@T2??<3(>1!CZ-Yep^t1vmS+ba;vz>~Y$a9izb=3ffB*8$kK&!0g?FL{Bq^ViaRJnxp%11&a4V4WiC25qo|1RaM`mI!'
            'fLSEwm^g9ZZ>}=&?w|n<vmQq)<oHo_z-%Y%msvbJ062LzD-X*2U>*SjUrNMWlQB_57ZJ)YG}L8=a3$xXIx-M%kYxhU5eSZ1jk3@R'
            '_7uzslG0GhL=`!{fPlDysTr!J0Kuq&*<IvUAoqZ1BVi{*0m<LB!f_F=3LT&+V9G1muQ<DanS#c-Z<BbA-Y~o_X{zQMW)mvFaf513'
            '{HQ6qG<N{CU}sqtieTKZh9Ep$pg|5KJwS|Nqec?Lv9%=lP`!$w(M5C_muw$F>q{y7(c<mYf?z9z#eiryHSMLvFP^u*9f_lcbWc}k'
            'Ha#=Nqu_viPEFLN-wM0&2O&5-e1jN%zSro3@+nFZ&29oj?m-)&!tpWBx->2z-)|TenjuaHOx6&;P(g7tyh$T750s>>^m`y$!G00f'
            '4meW0&^z93X9qyIkA~-_I!e-1EG2CypEXTp-vhm+umHND509x{mLn#Y_N3DKLCZw9dhONZF0WC^XwUo4q<5s5hhd_Q(fs)A=hILU'
            'o7Vf?yp&!s;Gya{dS;D|`_i3EQZF>$!2)GuyCdTvZYy@?9!kpTP|AchKU>+;jRY63*3|}^sZiLK;uqi(pc0UQ71W8dyuMhFP@KR{'
            '!d;0IN_%|<t))5YH>8mEwVVhdUqBy0=*q7K(gMcxcMQET&KO+r^F3+unu{@w%G4jW^uQ*dLA1tbfgUZ6#5JjPm3|DJy38T=Hp~A4'
            'GVJ%yPmhj{twz>SS;}6~G#6x$l&<oFH}up)C!e?)ai4*0CdVDqhR0Q*Ur|v>rHVBi)Mii=MUn|U1eiW&z>6688tehI1{_h*Q`n6r'
            'blj!mWIl!kLqEcJ2FOiEuB)V#VA`5l`N2`hE@QxpaVRcPK!LU?kVARqtYJXJW%Pnk&}s>)Di@qyKwJq>ka10$QU6?(fk#V>o*zk8'
            '$2Xe-RtbncrQWvGO|PcCk+8a(Yu<dVjyy#EQUos!9|z`fHBSmSWl5y7pnWag8AE%ATaA0j$2QJ|1U!q&H6BL@4b{|-=)7})!4#H&'
            '=|!=jawXcj>ny3HJ~PIiOJ`4b#3!<s4{SFGX24dkp@^!5YSIqH@fhea{x>+XCrzQ2WNJv3?9Ipme%ucQ|CZ431$Jr_lv%KormX;C'
            'bP<o{fea(MMv%k7(R_P&P0;9Bl$luOvlMn&>!6@hY9zPGRg|a=C0F@-;J}-`-5i!woZZ*Z7o^`!R}9F?WOTPhJ`6(sh6|m(Ut_3>'
            '57rV0rn9SB-Pb3r^lQkFIJ^C4sxX+QfKEm|MPL4$;|j|@-&ZRtPl}O6Fi}m2w7_^pCQ~4ljhkHuQ}9s?fx`><*#IxH8OZgK*w)6f'
            '2DNshndC)wNGN>FH={xuxjNpJXr*-C%*B^ILkBJeEUoNzJS|z-fhGbt(BH{=0@>Xw1x@~%<M~~^6aVY=%NKNhaOfC9@`ymVo{lMD'
            '9;M|*$Z$Afj@f_bKmP6CWpr9e%2k%0BJ^9*9`0z-Q1%R7_yd%DXWF;{U98g?a^|*4gmDYx6O?N5IW<J?Kp3m1g7jSFiaC9|kWd*T'
            'fQw0Mwg(Ox1oQC98rYf91hR*hCfglY)8D+{6`<ew9RI<<JHiF2+COiQb2mdC?pdnY-7I;>?vo=|+s|dYPpthW(HuKGp!mj~DhwS|'
            'Sj;nWnMBGWq*lGHFYu%D_aHtV#Pd0tsA&C#?BvE{1QTqFc_+70NH{HRbu8kNrXM9Yb%3%p2Ri6jvHHZuqFFD7c7WOoJw0R?lv1Xr'
            'ZQ_@qqLfyn^eWnvqzNb+#Q<)QN{Z0RmP-IBU=-P5lonsU5r{A(l9#l#VEqT;C2AnG;0@1prXxVe->sMB>>U17N*=?2SwjSjV`9yK'
            '44uV3rhUeVQjSZjOldVRAa@O&NE32I1iozrYoTX^$cOC;aUCdBABwZ<08wx{t~P7-!2$s+XFv2A8IqH%`d8@~$7*=v-8K}1!9Z<;'
            'KyDa7%|MA&Y-;AhTPpSjDFMd$qS(DI&<r25T@?DHZL)IhsS}6XuA-WV4@G<tzvqJ#?evR!6=hAN4cpWZN1&Z1h*ZZ0FQW2*#(9WO'
            'q|H6t(>}lCTzjYnz)Z8$K>%U{`<bA#sSJGv7TMJ)Fh$cF=mvtuhpMJ$j~dd;^Dv9pNg-M#o4mL^xidv;SEx=tn}}(a!=OPdhe=sV'
            'N!8>PP5$eHGLDMrqE|GC|F-<5_jY!tKWzT){ciYnfWLm%H%?syNekwM_nLRi*HCQ;|DFBsx7qnOzxyMWAMy_e6q`tH59%@L`&*zI'
            'O};f)rAi{7U!&9Jd$R}f{q5K<7d!HD6KxWaFY68AkHLW*-&ZGsTr%b^NDP-fH@auqA4>1s^nUl-uino3uu9h_;r(s@y?*Nb4nM%m'
            '81M#Yom%&f45ujC%m5U<oOC0y+;P9pEU9XncvYMiU0qTEfBlo2>&ppHnuHq(AN^}mJ!ys1$GEYG!`rq(b%Jf**(MyEqr2Ness=vr'
            'T}6rVT~=2yJSeeq<kaAKDNP~5ZVU+k&Ljcsi*Ll;2S-QZ4$ZEFj&C2}JC?j<Q+?Ta?RM0hCh{rlItt7#Y_}F0!u(^4@z}vxS2ghM'
            'XY8Pl3MS2#{ERh9@eWJA1Hz4bh&ooTP}8sd^FxuNTh0~kH_M5KiEP`EXP^&Xn7esC=GMu9KAZIsdJ(IiN^`GWXox^MtXAMPXs{g<'
            'uz+k5O6R7=5LM`6SE2WjIhelI%4;HlTnVu=5g<%iIzC5|TLKOP$r8CaWvEw|1PI3xqB4=b0AF!&5f>JGlaKfE@Ga3>`ww->PjZ37'
            'X4<+Hdo4vCX>}q}UKUE3NgXyt7(F|rv)r>a)HgE3X8-J!l<4d-2)9J(uyGqS-Es6w(=W%bfPqcxwa<ZYF}M#DBYNNQ88`mm$Rn-R'
            '#AKE+#+ZmQ0+D>9*db+wilqMn&MSJ*y38pt+!5~sT%mXtNZ*4<1Pm+SlF$Ttw*nci(Xfx7ut%~mV0JKzD->WEAz=Sa)0&*a*a=U`'
            'N4)p0B4*i!H*CMY<f(}(W$f3%ENpT^4swbt7Sj%vb8z#EDT%<y?%_!+Lno!EfSV5e>2q{tQ~OqJPv<D7T?&{=mOxfLB57(&T)a&V'
            'JGoOSs8CQK^qW#}@RX@3hXVB0W>$IrImih!n{1_ao~F5Z9Y}oTsg-duTS+46M5mYx+828U;In{yW!Y0WKm&jsLeVY?A{f~zJcVgI'
            '^a>3-J&YZtP<HX1INCE=Z*SX%iH<(2h_P!s{RDb>Fb~61uJf7jCqD3<Pc{T2Ja3v8sO~Ue7~3x^lSvvI-*}Fd2-AkX?YE<D<mxs+'
            '<c1o`QHIT)8LDNKK{Xa-Mr_un7&G;r6WO9%k_<Z2dgcxT*AZohh6&+boH;m!<>d3;t^Cn<v`3TCD%e`u-9@dX?k?(C?(V|v<!_gD'
            '=7&{O=aOB68ThhT(8vUc0v!I(sKL(8P07oUA4-y>$<(3V!saWvzQONO;FDym&kf1zlR+`}e7<+%!lGxb)$Kmfi5piL>sfv5TlB>O'
            '*&yRpD3A(TU(b%tc@Qj4^uvzNQo883F`&UM>e@WGzS_ROUKl~kH(+7!$_hMV@e9f5!g|j==zD^yZiVHRAY%$K^HpUbmR6@;HzE|2'
            ';g4EK*kuqlT49L<syEvzIBywodT#)Ch@%b->tMeVs_wg91e0T;L8}*gH}7l>n5^+tBmKLa8hVlF!tm^}YBCOz_Bp_fFwPLzqTXen'
            'cF!g59(k^jb+lBWNNpOy54R=~dtZx#FZTzay_T-|s348yh<mq4DX4g-RfHKf_P*9`KLJQizq}Z~I(_l<^p#k{u8x7H`?B3;n-8Jy'
            'xGdQA3tY%*twdt4b891+!+Q3B!wy?=v?0(JG3Hd6k@X+A;8i|T<w9c6xv(krmrY4sIRtrKl}yv5RN4|Uul1<X0lmS;KFasUM@L6C'
            'wg91N^a8`9li7B7Z7KzfR|JKAhu7TT@aQh4RWKcrr+|=SRMP?~lQVDGQxDQT1R2e9mQW}+kP0@qcS1ATl+mI|1K<qx4^aVhB|t_Z'
            '3{YPgNX4}1%?V!)Ktw3l$lF&N5R?bvHTt&72#;(dBsf7%mNH65_lDhqS9Mw?Yg8`7;-l9Q$_9YFD67B#wJ3}~8XX%QZWM7ObeG+r'
            '4)}1!y*^iN7pDIfxuanQm%W|t6IM8hq#-<+c0KH?)=lxu4Y>>a;~WiR#O2dVV$Lwl4fbzH9ohDG_k_|u$zYWMIT0HJ0-$4q;Ns3`'
            '+5z0Odjf*1$zs;oGUR{8DY<B!VHvx?m@2Djm5}yqFIHYwZe33r+MV}|L6!9iBhh8iBg>UXWip|mx;{M62{ps`z2E*!y}!fU1{D!^'
            '#IZZ6X8SwKlE;sN`d#PBv#}&Xvw;Ks(3}>J(-6XD!N8nT=!dnK-w^b6buYphOQrEmH`Jh8AOiMz;LIKl=C#*cgYcKzg*<8&Z?j!|'
            'z4z@FfA=MsMbMH?a!4_pEIoW_6Gg4=Y-|K<8nkFkGJ78)L`2FsgS9OvA4*L}1@?1vY;ZjUKn;uZz_a&;p5FRA-@o>rd;nXcLVpX#'
            '?RR2FL7JHUcE6($D#IkOql9wsYcvfq5F2atg14ERX{rQ!nVhMU>|zwu)%@ULplseMKjtK+&_o?6*o1*dp@4OAN=M`5&@#@d@fu@u'
            '>bN<xZt(dO@+=x-s832d3=LRircI;lqK+=)2g||!Cg@3)R2b~dsXvXhU;6BPqo<%0zEQKz&N(=V_8jO{<&!f_E;970hvSisdL4Ml'
            '`$oLX=ya8~anX}HXrVGcPd7Xo6e?;=#z=uYBCe)`z7teS=LjWSw3dX<W@Qo8C-Q-z9)pcniU^(IO6iCvM;w%_=a@CDH?Vr-QQj!Q'
            'Vf#oQfkNs1<))G&9do#}5&mhpeWG!iXL77&Qik;Y%0GWXBxv$f5Kh2)nO}Qtgu&Dxm&r691=Dp+nci0G8aD%oJEH*R<Y8bPxx{#k'
            'F~3qM#-zI(aZ`h4KiKg`NELj@0b7f*U1>~EnQbnknG8A`*;yLpaW*3_(lI5ag}&!?NeeiVN(+VYBlQr93CPJb+R=R*nvLS4p8Xru'
            'yyO<^%OIxa0;#9L{tO&iKCn)qazJd4p{X!u^xS3E+KgVO_21rOle}d5w8Z?q=+p_AIjh^A^f&U=9{NNU3#+{Y+P_#nxSh_;%}-(X'
            'evy2L+@v6)M@bp_kL>NKHQ&oRh4oI-Xc{dC#yC5aj%1MT$VbqvHE7V7^_=66RCIc`K}QQM4Tar!TM^=FKM;G5EnnT;RxfYU2%yD|'
            '_iVx`lmAKn9Dkk#J<c30R&kXjVv{fX2J0-D9#pxH!~+ZbSdv-TRMgs8GE9IyY-*e{lY)Dvz!;t$$ij-D#nZYe**C&p5OFCt+{ib~'
            'L(RS$U2wQ~DT`S%)@=I(cvjtr-uyl6`f=a+*1^%)fr&WAu*qlFFk)s8b(M2>Tl2{hc(+R6=U--<ELtT~mB+5^-G+I&frC(4AU~PG'
            'xh<P>D=?qI3f-0=Vv{8cb<MdZ)|jdp-m#IKa0ypfPPtAH%K9n{7EzYOm_Z>I&sigo0l}nO2lRfCTr5&lxzYs_FRcb5V3Qadq#ZFT'
            'TWMJ}^(1>Ffz*_oqUAE}5@4JA{*G9aze|SXyhXE(S5lM2i|nu<PI5N+sV%oE^&@>}o~N(rp8XNsS#H>0nmv+LJh~xMSE51l;PAjX'
            '_hygM=~J*EmDY!qqaE{T$%Jf?M`wn0(%V=%?^^zLH-;rK))9xud;`x_(@sA$U*X*O`iOOZJ7P%w)}@w(x_(E%DgLmX7!7AnFRd4f'
            'UT;?W%AVhCmYUxak9)Ucl}&}8v`$Buw4&PR`o@Tj&p6e{zl7qR2lTf8pU5rC<NuSmWw)CiO`klqQH>-|Zil~?@QpLvsY*b*YLmeI'
            'nJhXUEvp3h>^Q}%es;6yv-*R0{pyXlkvo?Xuu3r3sZz0KtCKPD%)X5`?K5WNId=AZOoKtZ9U5)_^ik&4^74RHj^wB;Z)pdT%2imz'
            '#4!nlhH=I=4Y6USf1`=04V$e+Eqjrtu4Ob*70{v+3a7K}3dgE^V>6;oZim;hNogfQc=R~9Hi6Qsz?@4?y4BOHsy&A)Yzs~WQr5dk'
            'wf^7#{Xc>6@*3au%r{$n(>LF|rOf+6YX-?*Rwsm}t;zC9+ejr}OMbZfm~0ue?jG&(!#go=@^Sx6ia1)1FQT>U+V|joV<c-j+L+3I'
            'kK<ChI>}j(MXOxo1-Z_$C+vCwhhh^$q1fALgy)T7A=#-IcSJ#Homq=+D55(oBdA^HDsG53@L}XFDbIZ^V4VSDvr<Z77OgNh99p^='
            '5E>vz+*F0m`@5Y5xGZa;#esY&ft?+lH`;6q49vI#K^1GkI5<96@SR9o$zQ^^5?ur;OiqO63bUfL02V6p8VjQ)w$}>=h^b%UU%$h@'
            'hWOW`z)3gC5qV?66Md0|MHDfG0k_zfcB0-flOT=M{Wjx*k?UB~S}Amk!TJxI`h@8I`g$q8lcw<dieGX!gTFqm!FS4xeyr+RJHlVl'
            'Kl2^8E=W8z=dsPD@djw@7CfCEt5>r>_MDe(<DhSM>}W_5LoQ*NkmNl+BoF~n_0carfic}r8f`G?`uM$~PRB<Nk9P6=y2eJ=_{|A='
            'HK;tSTDbOKZrA>!FC<a`zdb<$$u|M%l!GTbd*va>Uc$QrgnRvw9*6f1BoA%2E=96xw{Avi|Irs5ar!moShcH)eescb8hG5ee?`;~'
            'k6*t=k75)xIZA;-3@9|1aSPL|uJff*DPVHkIHjZ>1n4xvpc({#e$jD6IU8d>M6APtTy;2A4_jRL8c<4^Sr;OoOHP(k^B@coA=m>u'
            'y3CUqSKC*cSDslwb~lagB%bYycs`G(9KM{-8%@nJDk8e>GrHh>o)@fw5l^D^le#*fK*?2piN+*edG3LwE8)$BnK-)G0)rJ8=_dpA'
            'Mh#Ik-IE!x8|-By2MnG{rX5!-f%R3~`v>6MGck)+7s$YG@}-LGT5t*$Q_&fmy=GPJvuvK9n^!6Ed-RJNipzV0qoX^<*Lm<W?D1D|'
            '`7c934n7T$=x=@H{^XY1SmmI#D2->|!_>sZ<81bN5zX?eAz+qu>E*9Qw0@1AIYV(`zBixG{&??>phBfJo*zh=J`^t^yt@?dy~y+v'
            '>!W@y5+Es?Cxm~YcugFr@#1A&A%d!(jg%IA4>F2+{e7N-xcm&*?PMB?Uyu#PS^y8u>tzf;;1NgrFY@0%!N0seL-81MTG^m!N@zT~'
            '2T+0L2g)n4FT`8pfEqqm53kh2A472}ZQRzv$a#EBi2W<H5?*WiybWowG}Z0H3!8Sq3zryf!@vN~6cPV~|NP^(59Q;*ad<@k1G#|y'
            'yXGU>rc?dl#TLti_u;=23oINw@PBqQljI*dcif2#r$ouTi10kxpoH?8wH~La<dD<MD6(rpiJ*kyO{}jC<I`IbPRr$j2?NSQ??e7!'
            '>->46_w5`&vbb?hdn&n?ElB4Uo>HS~2gjPsxfwKLh~E<P<^#)??)TN01~;}_p$cajOc!$uglTS0gxh5K(QTGz4!HAf*1UaZQwMx2'
            '3{=r#x0vCi+XnizctK79fY744%5CWgjy8%j87SSjl7ayDFX1TnK*KEPoJDpmN;=Te%_HOXoIhj8oso<Pd&^kp0S0n;ekl*zd4vp)'
            'Iyw(HvA}d%uj-Up@G^!I9<NNptAqx7ehnlh67ry%l$}^J8MdeJ06!dmqVChHdG#-`sf6wBprfqgD7CRiwQkC;YD3>{Yia|iqqWw~'
            'Oao%>dK8<i`8<wIZk27ToTIs<5hF`SqFeONbTbO{@}~#?gxgc1B#ln4a~@n@zkNGxpAY~1_U+_&yS>)k-3B)TaZVix!e6gFN<UU~'
            'k4g8;p&x|0uHwN4;i=I*RrCF)NtEUnXe@=(%+(Q+rC{57kLH4H=(|zSIK<`o8eTV@_Ff-&lYgA#74OKQ^eL4*YQowMbN-v`BtML4'
            'oPwcA>Cw7CeKWtwRywY8r630V0wJ_+boKHP_H}i}y&Rr91{NQEN8S83*VD~!x|wd#eJS0-ypQf|c>Lhpxoks*K(NBOx5%sUDrajS'
            'K#bI-bY+}!01toEuc?*{iK06g@SrJ@S?sfY|D0enue`<r)6<}0Fpih3B2)=eaJy2j?V)vBvH~3H*dY*Su8gKbei>y^x+#-#j3CSp'
            'd|NWvHz=>=o>rH|SA5@A6K_9V<Yk=6+D1guigI*epOr^0Ju9tJs4iJkm!kJ|#QhI2-WPfG1M1&(60MBPg>#uSOX|EzzPL@uu%B5{'
            'F2%tCC$(DfQ4R9tG>xNznY)~ldUHO`P&yPwGP2lAuEpGBG$|5FP;9s#XqBs6&sAJSh{_H!$e2C8yoKC&WE8Kx5O_8<aJZ6PLT>o>'
            'Qu(U3OEp<WU<oLuY2+fP+10TwC))*#zeO!sO7)d>+T$DBY!!3>+|amFOjf3po_N`W%A5%#q7lG70}Y$W0gBNYQ>f;^pq9R_hwX6_'
            'IJcLfJ3I;ZeCXOXlaemP$$-|-NJ0#*ucVr_-@oA+-T6L9)JdA4swEP^HO<L30Pbazlu-&>?;=Tu;+jBzRcr?)4i7Lx8Jq(j>ZL%2'
            '6yOwteM@1|Q>*ixF@HPsC-Z%KV$s0KPusp|ElLI4aPM0G-eTIQ4RnTj?e@gQm+w4Fv9o6`fZnz92HJ10IQOyjzTH>V+3q8N^kKv!'
            'beHM-ZvERueH#klfkv5Yuhf-&n4-rfOu9q%`%meFd-gaDdq|sZy#4J{I^a9!Y+7k_$4hF=voGiyt7R8HvkRVo(vs~4NlqX1!gt*w'
            '<96!+wOQieMVlYU{M<l>2{e$m$g|`FUT-Isz_2BB(pD<pBT$D~G7z?u5bQv9+!*CD&XOz<m@!#In+QbiL138sn`EfY1dSMfjLc45'
            'EQWH~!A+^DIL5N6mqu*2W<w~VJ1>$yQ`Y-sw8^3+hlmSzD**8{)^aP(r0a!2whlCnE6>fn<{6ZYkGC6Mi}th*7Z`4<P{**{i%qaY'
            '$p^XMJs);_h>!$cx7ZLWVjSw*GQivNmPvuP`XwdIoO5+Z;LM~li+iZ+L5nu4scob!dsFP1K??u<^f{Xc?Y+@6H>{c@;2m*;HFE$6'
            'xnz$1PcQ%AqEcXG^7>Q`+w7}x_lSL6j`j=Ek=;Ebr<q^E;df)ERFuh`-<##Bw}z9dT|*Cs1gcq5=x2TXnebnc)xhNB??zT8Whzx)'
            'jMPlZCc0<Y{$Y){IceWx2UR^Gb7`t*?pSN;>q4$o^fyPL`SBLc+Pd^dcxLN5wi`C3;;8b#P94dK-N^i)3K{ljwkCS2(DmOAT>a6p'
            'EqC~TTcw++yrHz;?0Z8cpRM<XDvpKPE^U=EY}PlzG|{1>@NHB9&b>droYmwuM#<?V%b<M!c(<`81vBWreD&=2&t5!!&dz?sy=ZSh'
            'GtX91D)Aaf8s6WBeq{%WE;RQ)YLo6vs?|sbQ4Yj2R)f#-n3eXF-5Mz5^XEUI2>j;FYaA7hyBLr|zT0!UHw>@*tSe6RAIc2GadXE8'
            '4<acI2S=>m1UVY90_?ZXS8L0pb9<&fk{aZ$=$mdBzD)PazI9TE-}-FZra3PehcC(7YwVkBbg~grvW#EUY5KGL+>;y+x&IH8$3J6;'
            'I(nrfuW2Z0F-o1_Mim?AOm{BT?RC=Qutn>UApfOKig+wZr7vN&+n1d})=!L;aF`~mgs#plIA694WvfIc%?iUpFDd@O?zPlG*bFlj'
            'mbw5npvgvoxK>;>l{z1)y+TQ^bLGYFPR3baya~>?_V`r?<&7IoqdVrrVYo<Mk0}&uIl8uBZ`;B-x*!1SxXG99-O4c4@g?pcteHjy'
            'PBGn4)pNSm-@4bVIdiG!Tvs`>x^=#1>?XRiTjrc=-t?UkgVhrL?yV#AZBa2Hq{SQOOXKPm-c0Dme-V@XMH$5tO7tH5J^>1p0`-$u'
            'o@bbYS?Pa1F-Gx|xhzS|ZN|{AGVbK8T3X{>`eJhv(jh%Y8=dQ?X=c7Pm9%}%G2QO9q2HncJ~>#C+;tVG`+aUv3vcdJL6o|NG^|<N'
            '2D+-$X=Tix^|@#L3hlw$;N)fG^70M00ZQYB-vga;c?TD!HumdZzJK%V&GXaouRlI}{qyOo@lP+Go<4tl>kS^@z29_wDBm!u4=iP#'
            'rY0#>ba!XIDz=XtQF|UVmy)WT-qF|oHfUX}_?H^HH5<2hN@+N4zsF@Heo%-0sB6|6seK1j#Z3;)+-i(9PSbv^dV_?zyv8kWhja@0'
            '%Z_2T+n2Fn-3<3w$5}Ya6)h<Hx=9(NTaEGkFKVrEZpK`QLXS<g!2mC~r##ZVA~=f<KAc~lpbPzu`R)&IHwQaF+-4R6Wqs;aOLc9G'
            '{dRX4LBwUeiA1tY1n}SUG!YXxVbo@U0|4)ksYMPG0^i&-u$LEEyc}X|6&@KCoDYvsuFs-ublqwZwv~GpHtQz|3dRgFUOH69Ma!mq'
            '3orMqb<k3jL8qE`XymjTIt%pI;M@aBMuDTdR!j8i28UmjpM!>H3Sma?T-nSE)h*=qwUN~f$hH9Ig=-gap=xo?nStiL52pUwnt>+K'
            'SQ{R{?}A5PJ@8mGY=M)n{m}Sh3s~c{GGcSR7bP2`e#p+teFuzfz}L<@USN(7KuBKA_NhzggqaL+qr@3rZEBz3RHElQl1*ihI244L'
            'Wzgt>Erm^!gn=&VK@kPo4AswRG1t#@%Q(^}XrqWQmJfQ4@jgd``yNX55dce4n~39w@Sn`<AcvqXRKUQ96OP0%l~v*!3szqo9*Prq'
            'd1CR(`dWoH6#1ze%wFUDZkUVh3D%zntM$FW<he10V=qwBh?E4DahaqBo9(^DI5b_I59snwW(pPa+&8Z|v6<4~pfZaq{@V0WIaoz&'
            'q~%A$%i$5-n4(`q!=vAsUZ)4sA>AdFCX@Fk5AIQ6UQ~R<4BiqIpkzUpNS!dmb@F~ZU(uIMeR=QXXzhTF@KY2`WoPYKR=lmDz2Qj>'
            'J(tvu?146mrt_hc@y8}bak|5<Hmy6-$tJexNhQXvoW302E-yvSn(ldW68CR?FS{p*SYewqxGZ4=6{?3y5iJvTtA8YvS>}M=r`0C<'
            'kRa1sqoOG(qgC`_ph$+DRb?b@EzdY7wJC*ar5U2YOJyS}k!#9yOt0*h+$L5iv6Gyt9i!Vy8?tAIvNWa_Csemv<J=&xCti1loYB<N'
            'LA%}TW4PCma>2(FIXWcL6JmP3o%rF-wpxA6tp}$}m7oEpxpq#k6eYd$(!WLn3PqUy+4=rL4-^Y4_|8NQ-=pMq#v|X8sBpm(Gt^v_'
            'c&)2#^{`4H2}Ob)5H`we`wG8-gjJ}c9bXjDdZEMV^=nSCK_7dDX(hDIsL(!a+_^)4$lY_NFm{0a80&s*6juIG<`p`YO@e7n+a5t$'
            'YA0H?1GYcLZm*S{A@mH*UYRDWW@H}NMo`{nWSzm`UMp-4ne$Pb!w;LFx>1{Hx8W$RZ`9%)&ETds<Lg3E5l?NF)-LJt>m#oeVArvE'
            '@sWqOx65l*bkycyog@JkpE>H_RlP}D>Itu^^vK3jlb`r4l33msy*4HoS<Z6OHzb8Cl@VvFa)rq<fv4U$@3N^6sSFeu1=k%t(@^@`'
            'V4rfa>+YEM#I{DxWI|Dc`TiMQkvu!c)d{X!Q?m`O-TrQecx55ZIZEGWmw8>C@3))aG-+fLRGZ%1oU{RTJfL~cgkzQel*O{?um&S3'
            'i96i^2AjRc9e3R_>TW|T{%q6)g@#t(*~o~ja@B4|u-NFXsx~?aW#E$0s@lKXk6b)Ga{m$nI<eav_CS`PAKTy-%csx1oy*tUgTbpY'
            'p3=p<?-9ZyrEIoOqtF@Bx#G!)8{}II{Raz|py1V(SR$D)W_^JJn+H!5yy$!x&5G#H;*LH%t9Xg#r34rSaxyd|Iyyh;7-9n(av11a'
            'PHz8=qbyQB96P-p;9tF4OFc=4U;v-&3B(~-FFcYCuVPEoxwWL@h_g!Ap-W4&Xxsl-xd%FvU(VC$f*9dDC`rj&?*zHE%%3e4NhC?p'
            'm1eq;KEdYVD6cMBxIGWnNSmm%A3aHy72!#ikbi3dB8SJ6B%5vOqOH}rM(n2Cm|+|1ck*C{&x~^$#prn8NmDZ7RPCl$(>LNI6vucC'
            'xyfp`26Lt+g(i(}#(*C*t66?U_nChHwrx;XQ*>m^nR4_x!Qll^HD4$dNNtU@A<qIf3bd-o`!CMr(S{j`;pgrK#?qX0ZV~)IuI3Wx'
            'dy(TU+DHSc<dJmVyNoq`NU}1*Si5PALIWmUikGa~Fg>tu>TT09YNbh2J&P6*NM;|>NZ}UWW{cvg>{o4Mc*@gR9qbwIjyB4!02k*m'
            'D=DP2hkAty@efI1V{5cV-wI4Sq>|{N=F$ieYB^zX4Z44%!?eyTDis2%bZAccbV05n#ypMnr6xP3P=yMysE(^{cl(Sq;HZ3H<~gTO'
            'AI}g6n0}_@NIn!L^X0j5*6;wn3_{Y$*kn{ooE)Y`+BNAi+@xmf_uEN$Ok6WO{W}0SzW+uHbOdKFT4J2;ZLHlPGtSkkUV9XMICyos'
            'R{IMki`#Q!80{&r?{nqSAiv=X%@fQeE_+6_siRM3uYZe*+50qI=4&}?PX8RS`b&<ZBg{_jh@pjFq5EBtsG(~d%ndaLkOW{SX0<_@'
            ';3Q&_0mS4YU)4Y$9{I^toY@5HV~Lj48%v)i^QI~aCXdBcG=ol{(+;Xx3@+9fJ8VCY>F`>6+WmHR&mhU*{KJ`Z@Ah+?M)|thp8MKK'
            'S{QgeN@ZQ|Hy41{ZD?q_^n_lQcE>q{2%YUN;;m-5T3|b^m&Tof#<VGt*PghB2EsQFWpsMcqOyp@?mK@UXfI#1d!Rf6)ieYhuElP$'
            'p}<f`HOGW!=1ijxUXo?}bV{1)8cdIAYHz$H_3hC+^f`gar!^h*LA7|erl=R{4y`L9#%*-6HXLn@*EW3WL8m?c7I0YSqEP$tlc31x'
            '$6=@2Ko<mCi9D{En;<i%PoeDYY)ik0*TR^;JIX3kWHWz$zoBMU>AhlPI^pSg>;fle_twyu%INcD2Ux4*ZphNki`wHdvEXV;v|FSL'
            'VB?@L^!PdKswDCMCgrGINmBh?YC0LTwX<o3sa<eXAa4hdmczqP+(LyK_+1~&%3a#jpyk|P*6L6Q2Q3c<w~|LU9Jqc8b{(mf_E#};'
            '3f!di)s<SQ25wXUD_+JtdPw6@TzVK0J_LZPsLk71?a!9l1j^WI6mMBntPD+m0q;rLoQSQ%v`sC&L~>fkRwf6wH`!(QS_7kyywAaD'
            'u&7L?$)jX-Dkx2PtAbXcn~s#tYvZzL&1JjH3ExnKOG1iYK0eLK;1W(XztR$K%JhD4$75KGXpbQaqs&j}sOMLSc8$Sjoie&&zIHp('
            'A;KYF*YU}L&?#{t0(5<AF^I~f+!)1&j-lC$)U4WSV>F9MMk<5hPW7O-o=hTg`~WK(2}RmK$W-FKy#?wSnn91o!q4Uzib$Pyz5OW^'
            'Cw>2nE$UUI)Gt2P-cZb_%cIq+@nj=!dN*H2m3nCdLRX&BQ=WXL@!+E=ZPm4yf8Zc0)x_--gmxYn)R{Y%$KISfh4HWT`Z7<ZG5W#F'
            'FWtE~Td{~PZ1=4O?Xe4cav6PG&|-J6Vs|^;QH;By7iaZ!38&*Yrc5U@`)hY&&hbfn$0hKP7~t1rF}-`=ymra`j{52D7O-PXcbXax'
            'cSl?9jI!)*=~m&Fw+q7SZsdmOydPoF4$}A%(~#iZdTD$B`#|CgunASM5$ha#qc2&esyi=QJ~|q2fVkC@n4H8{kz~y9egZBA9LReh'
            '-&1*qt`!`p6Z#mjcA>iOIWC)nn}g7>E^PgD&$`Y^-33_0WT7lEs7@LG#2qoAoInkvJ7Sf*N97X7>~WeNjDTZn30->)n?q@l(~Y*T'
            'aL65#@{mTaZu+xh_{K^n&QVcF*l_%`8Zx$#Qv|YRikx=W^(0No1@<f3RFv4SXDY8K3`t%v*q1;aOJc2C7;>7Sd7Y7tNj!P{;>F81'
            'h!=qAm_`m2fun|$V`8GDf?d6gd2zB_Yi$IecTHYt;xWvATrOZHbc`Y!Ffa=*UwPQlWfQm_y01xnHD5ubr&SmhEG&Tv(MddwfZc*#'
            'O!C<V{q@8Z#uY$&2}f~mK==_q11jWH5f<7Q%z|>Qm7J|Ongaq}QOd>3B%;di(j2V<8j00N<pUrv;LIGRvDbeW$~;$cJiPL+e|&rZ'
            'N0Z7+jIVeeI2#uHq(c;aS875WbkTv%KYsC4M1)tktiCBQ?t`k=(zQ`91W;DP4xuhyQ{xVWJ|&uW?2e`0A7D~5K_MD)Bn+cZv8ZL0'
            'q^XE8=MLqSV!WQBAHhu2xZXT_db*m<yc37U`<CYJ9a1#8;VD!uwv0m1a=b<qR2R1M?7@p^M9C+0=I5(LL`gSc>!C?vMmK1gVXj|`'
            'muq>$c0M5?tKd*$X}IbiO3%3d0N4t?4ZfrmK?4o#Hp>1SR}!3R8M~#EOHblOgipmIj!A|g=4nh|C}}kA!6n`t-URi2t;j3PIA+tt'
            'k&zjRW(E6(y7VfJW|Ws2F}GagSJWX<u9FNX;oyr!^8U9Eiex_t)Y@xKbqt$4gZVNNb(&GIjOmW)@GH}FXymx^18%N2cA*k^S5)<S'
            'XPetV^i-3ag{i;aZ}nDh*)QGU25?88J6iU4+dhBI`+!HCV`&NnmY=N`4BA-X8=<K8PEB7Dqpn0l7{y$sFd<0+8*i4E8@knq@&v7-'
            '83s4{+Q9r{t*CEln30#y`B&U7Yh~K=8F<?Hl?-ooBXGQ(2a2n=?kH<Tw)yc^25Y<h9Z-`L!@b`fv2FybbDa7og{Ygu_}oZ!?--r1'
            'b)&t%R`mMdGs4%6?#U>2y7Fk1lpJ0;Phw0gU%{bMpkaPQDU_QjihzaMg-ukyA9~Y?@i)^T9ho=cP5f8kNn>>5k7vA$CZ$RY-Q0gO'
            '>Em@UdIWn0g6VOx6T_VFYEIVy`Vu6z@;qXnjk)UUv!DfjcSb-D9tE>#nNXGp%3Ww)uK?JPcu)!0f_hVzs~XO`p~~=JbfF<F|NUb8'
            '{}KN7Hwu6ILlcN3rpXd8?;p``e`nEew+MW@Ip&Se9o8tKid@~bSXwb>7SGZYW`5;MN2Wc_FyYY(9d?vc4~Iw@2z<+cwC2)CAJ-Oo'
            '2fiyB?H|!uUm!Z`bHgS&LMHx-;S!$_Dq&;HUoAv}k=kA@erGOKo0hZ`?xCUR4tJ>4+luu5=LK1W<{18yBQEZcKjUY`T~tS3KHfro'
            '@=CSlx%MRjGtMG@^ItSF!^+b1w~5fGj{Xsu@jo~+Lr(o6FULW1owbhGNH}Ce^tka+g^JHdK#j$-7q1_`5HDXm{?jw@^z`@7o<Du|'
            '%DT6WG)=(kr65UvK=0_a$g^kzScA!dFi!J}$9CK>p1)$Ghil0ulSAt|m0NP3$XGJP>8UGb`t6HQSik;@vX|Iq5@eMl@%-_d(^uNy'
            'Fwo8t+UWr45};ctfnlz*3E3XegfuDXu9QgW@JV@}i!{$J8e<FQ0FBc*UR+Do9ZFW3rV?O<mJ4ZrLn{{L^~7sVC<&-EvN%Q($3W7P'
            '<`D^zjU;%ZjA~A;OJCbfl)Z#aJ1wipjZryrK@69V-TL&dP@Kkp!3U~K*?luK#Hz8>q>Qy&kgXBC1*REB6Fc@ClhJndQfi_aEax8k'
            '@ZTGe@Lw}L$<)a2=!0?HkrT^g5@Ywfq`_Fsjo@eq(jy{{z{jmIx5_MrEyg<gByGYwY|YS3@l-c-rswmc!mQ}gD#h)BK`^cU$^bql'
            '{?A@KIemWm)9LGH;``G-Jb(75m#^F_2YMVkQ_~SJ15N^yEaq7e@o=L}!71cmJA-r_icX_6=@bDC@!FHVFJv2i&0MswwY9!!xPuAf'
            '2jV4mCeNlA%L7s-olzU0Cd<66Q`UI5fIBJSKO-W^)3Lu*43EzvOF5T%K6O^NUAJ5a_ScUM1G4i4|BV8In#pe)AcJ9-Y4X2cKv8#G'
            '(3gq@`f~9<9~blEXbI4j($O32bZ0jd2T*l~R>5)qbW{vb*M0R-p6{RI?cVL+-sRMO8yEGzA3x;Ja4r5QKj#0De#p}l-H$9k9qp?5'
            'B0tF2+LcMWQDxF~i7zq0MU;2l1DfVFI?B;ifZBA-vK)#hp;*S7Q0tIEz(V&>GU$rIV4!QThMH@zrm2c&QsEMcZ<LxkxCJP(SurG8'
            'mT=rD{r>@s!cBh'
        ),
    },
    'video_analysis': {
        'sha256': '57349f8d512413efa78af89f6bfb81aa66a92b04d2b91f42800b68e7308efc93',
        'encoding': 'zlib+b85',
        'data': (
            'c-pmnYm?iylIVB+3LK^C4jD_b#)-3Ml_<A-?8%916FU_<$vt=O6hn#7h!I6<`IxahJ5~G3_RFpNzivMO5CBQdB%9-^WM)X9(P%XK'
            'g+_Bc9{(ttDqdw$+@y&trg0u;$11JFQ?V`T>Au*@ycUV9<+@IbTx7*{x)#+oE@dKC$I<*eqK=X77(qAgHA@f2d?n&M5%YO;ESkDm'
            '$tZ5>Vjm-{XdhR#EV<ciG<s9ia#a*NF+UeODG!xM>q->2x!5;Zole&PDNW*9ij~aQ+kIT_()?Qd^jZY#Vt<HB{Hu~TvW&CQFWa;_'
            '$WrW!L}sE&Kgwz%s<q6enoLo`1H`R3ATn_#zWeTbXi=21te`^z!yM^nI2o;)TGU%9;&t6X%Q(9(%Cz3@5ycjpN=&ED7yuW>Xp*#;'
            '9%Q^59nv+T%;O@wtXKm$o?ew+BNi6b5gDA*rYNCFjL@_6MvAR0rKpP0s;IXjS5&J6fXWD}`~Y~0D&9-6hUtqm2Rg?v(~a22X<oyB'
            '2sUXBS-J*#PDW{-$b*Fc<#l$PCS|&*vBkPLq_Tpgc@NY?QZ~RRIZ{|a-y%+6c}j6;R#{qYWd+kZ)?4WKC6cAcv!f^_Hj33miBm?K'
            '6rdmlZh?l-i{Bked6UXpvCh&1avx5llyy_)z-Vbc-OGJZ9tEs_EULO}U?oa98@+@{<AMc-4xs?vgs#=0$g31N8|X)4;go>95tFx?'
            '{Z;`(*M&&ZYBw8=$K%naEcPObHVp!gB9ZP7MOg!~0G)WY8jaMmQtIETW_2iwwX7=rq)zveVIAUno24ra=m+?dUmzc2GyO*YJ)zDY'
            'i`;<U#ub9le<IQbiM-h#jv}r^ei)73y?^okr+3lMFTVTf;+<HEyODr@WBCF0L5eeuv++cX6Ij2jIN*cmki~1cEwV(G_==YxD&@a7'
            'DeUj@gdxNSnBNWlEylo0&1RFXQ|yk5fe$w^EGfN)hDk$=AK}s;0mMCE_R|kPy#4Y0#jEJu+n;`Xc@h2m`qjnT=;D99zj*WR_1iZV'
            'wzK`=a|ATo7dQNK$3Ji7YEQq!Hz~jP{Dz@_^Y$f7`^SsFD@eh(s^hZ8M*B2LGKs$dbK{S7Q-SyyhojLbksIWaBD;~T@G9f{S_Yqj'
            's3@9pEu*9<SyokYJpo37{xGp?c(Qy0Ys3T^_=lX_0@$aji(nigLb84QAS7-Ncr51=4>mr0_W3*%)2~GhJ5ye6V0YGlF#fn&FlkZN'
            '({y1!Zkv6aPd6njV*=YWEAs0p?M}h7!)*cbw#ZZ5n*hJa6BGmp=;A|K(O%M9jwBCmLX_ILann+DnXcjrwr^ZsH@xM<d)U@5fi*z>'
            '60n!BrwZ7kFqn~36(A*k{ne)SufK{z)_@F3^9E!F!=c3zAerLqHa=D(CRMyYz>t8yC9rul;fEM!pmfCfTtgb;$S@HYRPJEA!pn=d'
            '-^?_{=x4fdRvz#a0Walv94^el(E<Vg5zzy8%}`FxL(^2RoZH%7Pwaq8YPtXtJiu3CjxP;}ua;uIaJZ+8(@Kh;<E)VvWm%NL*hA)P'
            '=puj*x}$!Lz_WdvrQ>#J7Q;OJ#0=a^8l)tU$JvrCavj7r<5ila`CcAn?f_bdJNwVSl=ljsx;$DGcQ>h_X3yreXY=ok1V&4u2UY<%'
            'Bov>DFXpp3b(SC2@=%N5{qex)PpqkgD21WLiOG(6gH@nfsuanOQYZbWA5`%%t$<fG`330-0I3P>&(>Y|?r_b=CTEo^iX^^H1xW2&'
            '?BZh{?}UR3FG6}ON=4;#(?_O<26X3>#JhAC!*t>j#+#odu)|l)vDvu{W-S5`++Vs$4<2DJ=Pj2)$^Gt*G%J|(^-@Zr+s#KtObxan'
            'SSZg<LPdqs1vv~+fvRNN$>H@L3FGMzhgtss!}c#~@e%B25aSsLTd@3jRUc$mUt;~yQF3_=)Y%{x?c<M@Owyzvo?*eXk3R%3{mBU;'
            'hYUaow6bXOBxt)Km~bLKga7zd%Y~n5iy<Q1qjVF2-dIb}dr1IB&o!`oMfw*N1J8E7mFpdt%1d!A!FsGq(-2Lq4j3>`u&?26)xD*h'
            'dJwdzG<l>f&xl_p!Fr!egh8m){2&TqfRJX1P+i3B77g)jk+YY9>T=E6_zawhq5!iIW)E@-*z5p27hlh`Mnn+E&BpSb{Q%8#9=)m)'
            '@b#AV%hx|#&~sUq-RHIZP%qz?jqH)|j3c(69f~XqLR9xHiS@setKzgo>v~pYV5q>NpWFO|NNXfvZZszNmHQ?qCn@Oa!*(Js!T-#_'
            '<x76x2WA=8Gj5E2*u)zow6)6Lz)-rNUs3y+p$G&x$-w3=4jtJ*EpOwr4seRN0H1}vB(N}k&y6lV9MID50kun!+%6&m<Urbc2DRFP'
            'q-6s8k0?n?vqSj_`bbN#q;U73gVZrsku%f$06sWtty^%t0e8WE3i56&(5cMP<Awc>4l1~wO|A^q7x3r<`nuqi{`zZYY_KhF!Sn=P'
            'lenk{XepDKcniapw<xG^{9N9G79wz96;I{<P#>u=xM5&!GGd@!J@_zN01rG{$xN<mg!|rwR-#W04KQMW%Z#@5S_FIL3?fc?ZXpK_'
            'w6%>U^8uz$p$SW_0XjG>v6A5UvXz0pO~<C~5Mp8Riz3&ORwU>&b?j0!`gD&OI8lgi5cY}uZ{W?FqW%V@tF{3D%{E9@P1e9v<0J(O'
            'P{X_*JHwfwKw#HF2tOi>?>xp3_D3tKfnn4Z*RPn5v|dTeUb?_@BfQC@FwK!D@WZCbBk8dNccmyBmo6<}M45Gsj*V4-+GV?Ikq<EZ'
            'LB9Zd+BgBf5mGj7v0b}_<_)qPy&~uT!z?RqQ4t};2Yve<Anv+>&p~ajY~r;OvWzIR-N3%W5WsH*S&aB7co2^;e31AK^K8a8p~-ko'
            'q-#F6t1=U>&`90EMuiRAe2nut1;ZB?OCx<hHsIBwDJDY$q@C^ryy)W+%$-#d6S)>5@Tf+zA<CgCy+F~RR|F<G`b4}#K`8eH>XPYg'
            'Q6^$5o0269SbZqZK<8hl7;nt2sH+Y*q6VFfF1~;J9taox^7hA9R)nV77V907pMHVn`1?xMWqfRZ<4_ao5dD+mLkW2PTlI^HFWtrz'
            'ecGhgTZ-C%y9T7e-|DjA<`i_qU*c6kx)5fjyJy7>g?P3=J{^q$7lgu5NEI3mLk0BPDk};`rHro;2#m5V!J^RM(rujc=v6`UEV5?L'
            'KfxZP=kZNk;V;+e$Bcqe0DCQ|F`BR%QZ2Vxx+6q(F^wdD1I53km{}d;WHEk4iW6+0GG5maxuC!d5rvI|&1Qd)*U>5kkGQHqbC&U~'
            'C59)=h!JQ*^*42E|2`E}5+)$95Gr0|d(7scHl!#{<=MEPPF^;3B`yuDNelLg<A4iP*~P}20f>;g5V2CMCHfK{VS|4=d-lcWPhY%v'
            'Nihng2NjJlo_zkr?%R)I54NlL<;BnJd(^-H)SQ5!)Pv@EwJuXKs-AKn-l#+IBT9ivfN6~Ti6j7-7SOH9(Z)$e$}&WY5bO$G459eT'
            '7{l%oXw~38W!d8{j&A8b4SQhD4snSGD*GJ@Z2qGnlX(I}fqfknJIjK00sfB8BNJ={`_)k-Egu5wPHCf0w}4}`iu0T|)-)@wGmI@0'
            'qa!uPE&Gk98+0IO*F3qF`Jtv?r?`87uLGAULc3#CsWyhMLhM<qrw)(V{r2$zhnO#J7IPZQJct+b|LNd%v!QJbhLv2>;s@#N(`VtQ'
            '&lc*hE&b*6SuZFqou#V}z^XLB0d@&X4m3bFaaHr<^h31S)9=UT=JRLsg9nJ@K^(8Cb$nB#NyL6H7}89qeY}&=M_Cp;k>aK$CJv+G'
            '-Ig;``usPsRBS2f@L*|nHfetIdqoNu_c+BPMD(V~vLDoIWfUj}u-^q$3^Y*#8KcPahs0lc;z{;y1Y<~BLFp<LAcrgH+hp1fB6z$#'
            'FQ=c)=kvvtF^G5%{P}FG8P+E+a#7x4Gp8$Sx$`doQC?|U@N@cIF~MQIkE-)8=*R*`@kE2Yh^JP-O#V+>0u0`vuw19RxJ(~h0a9sN'
            '`qH5<Dl<ll^%R}UlKn@D4K2C~gg;U}kvHkOeY`$2;~_nU=e|u1yX1IiJAg*{>v*E9t0>Jk#R4q-B7=9|#K6!)z+63N^xJzgjY_mL'
            'U`?^16YS|NA2x$Zp)o5FY+%bo7}$Vyl-G3j!E10!hqp3bZ}kh@aCc`$xSdVJnFRf>enlAgGrqZwfHMKd)frFVEH5G$Rj#)YKNV+a'
            '>Q!e8ap}No!BGSFf$d`q`qsy@`>T7`GbCNg6N}Wu^dY4>RR12=j<*>fuP>c9LA$nkF_-*f;w;$G{)4HbX%(S!G7-3T7~?~DvzDC#'
            'N6@uv0=p37hfRDf)lu6@#(P3!`5a88I6=P#AhY%}Q1AulMKGG-UA4R$gM|sbz?QcU5%$%1A)e3Y_Y;9+jBt+tmG_6r1BVA+DRn7u'
            'Yr@QnrXproGXL>`f0iPunbkS`v!t^oBg(!jV+|R8%UwFR`@b)YW(yD-_AEYN#z3I{R<S#nByK_Do5|}LJ_8v6Rv0*p3J)6x3PpJB'
            'oM}X5aoe4W4Z?~S;>x(~bszGB@aPT)qn(1sXqQ)BpaQVLmopA9t7xN8{bzVG*&BgW3ty*sBb^rpBTcZoPQS6)tiWL$dFBXziwK9M'
            'yFJy`?&}AyhI-+9=t9lZ7)17-rk1Ga8^n4|N1U@cjQP0-=)S`}CkQe~3|zJ$S`5$uyY!P`jNAsTea7=zSyHzLwDq=U)?-&b_w;ug'
            '^noBYRVzg_Yv12nCv~l&^8;9p3StPtqqi2BV^W}jKC3`*6YDJ<cH<hu53`-&&u;R9=@a!aPlv^r9s|QVAST3L{d{&mIyq})l5kp9'
            'L^)bjVEfTJDXcX+U^7E+4ohd?CSCSqR$SQ!ES31-_zoezB3L56jxnP_w8PA-Vb0V5XE>j;IEfEv@1L<Q)^<cpBcs<GSPr&g_K=a_'
            'jE`#O=x>@K52lqjq4qtvzt^Z*@wkRPuHH^FlO(6Rzfc&yQZFW$x`7kW57;2k{i1b-GWgEWapAW|pGU!jmp=IW8E{0Pp&4Fxz;^`k'
            '>HLB!-cn{>B6k3x0rOQu`%8H*f|ZKiErddOA~b%X_A1yNSz6Q!T4#Vv5T&`0lF^=t4ef2(2R%(@agszDo4}rA&<AnW!^d_rVw$Db'
            '2k05KuY*vp*G5DtzlTna&@Bj0oI$HC#=BN`VBiW$*g1FVIoRre(gHw(<TxM#RQuiynqCHGJgpSW0~Gnn)Zmk0(%QL2oGT>{v<0ec'
            'NEz=c$4jY;Lj(+gXDmx`KA+%mbCdu8G>@zx8NQu8KM`%)0wCb7YYgWywCCr~NVFvBx;9eeJIblFybLhEsu<4}DWi#^ULgh2nGBrs'
            'bazI!C>Z|vVU{OxS;j~32FiWs5zUYt1d1WESAY${s-4YdSNH0mgQD!Kqu^60*fTm;G{GdQyb$$mAqW~}TcR@r1oo+8WRj90;Kr9l'
            '74t#@J{9f_^>Bvyq(R4^ugWEP%*r<2S5}XWj425eo(9Jyz-y_zY<~?e=WGZPk?ihh@q=MrkSQMkp<+(|gMI<P40-sf90JVqKq}zg'
            'jnmAtQq$xEvK;}wMw$}T%f(`Peq|l;Vt+~nU0S=Ux`f_W&e;rj$3eWRg0{m{fc?I|9P70lUxk?4<u3yx11%978(q)zwO<+ItOc$$'
            'xEY^QCB`LrMz(PSPZ<4)_#(V_CV{kdP{3pv-ewLis}d8YJ6I7FtpgIA)uKxUTT<{05JmdgBD}}niHG4YdlVg$$D|rOoVh`TqKU<S'
            'z*^z5R<yBH1K20z))EaEg{-&*=1H%&W^K9Y4AWMSzubo~R%lcmb(<mx@s&82)AMH?>kd%BKm#$XLK0==b7%hQ4THkGrJ#izn$w2^'
            'o;h%0bhN^yjLTG3LHCG286W!ZlLri<xRGTBoCL0JWSlj&rcmx(J<y#6c8iYc%C^_UoNOB({c%VEnUx|0Gg0Q1d)@&n^~G%d5Ms(~'
            '2er*pvwF{GpFMw|`v$i1Dqin~0Q}4IfmGfJsf*U?yr6XhGm-8sB$&sL0Ke!23Ch$5Q+>bZZ2p}5bF%f)S|vxkqp9%WiCQi`Aylxy'
            '$V&5d)_@KWo3yNI&R}5ocpo46Xl4&x2^?kp<J@lJl5|!qav7I+m4YrtsKo?f&8a<fUje3okPrv~&=M43%)~D^KuvKoihgK-Ehv~j'
            '#tsc#1<)1SHkC}+pr#897>l8Z108^72?3H-J=KuJN^WBe*OW}8byL?mH405W_?;(Nigi|0bl+rK)cRyA&P8F8My`t@!P^ExU50UD'
            'Y<AM4PCsK&(BWw%5cGL%WD?(WKNKa(r?d`8hPorIxw8w=hNJxp2g=Ced+On48hF!CoBh)A=0P`2_1n(&v1~>!RMjU!oD@TTM(%<&'
            'T4+bgtgyKn8m8#s53_I6{k%!&=mJdP$v=AvdaR)ds9hY{36PimT7K#*l+uT~XUhVYHe0m4kT_Rr#arP^*{hUlaYqZ?QtQ*8%LXks'
            '>CNS^6OqVlJ7b%-qj5m=UY|CpAgY7SrRnbjvzh^U&6}^5F7>pfmVqY3wfXNs|9WXt_^KE2qau9IghCGUb|*?vUY*w4LH4Lvpl=o-'
            'CrS5Ak6@b2W0P^}u-3E(`>(VaGoV8)gEG$2>s%&=PdmUyUK&CT*vM;~&)BqexZ*hO$&vPlF%0NWhVkyxykU3(6~DuH)MlS|mSXDc'
            'u|Gg8-kM#=twW8{!9p*mDL$G?YU?|)&PMsk<H+P=F+ikG#4EYM`QBju<_)l3O!gn)`;>(ZqehYvO#m_<*cv6R%XW=|#umv;yoR<-'
            'RjZqbWHr>rBvEDq>S%3SXmf;haED`n8qTwcr)<Mf4+(+ItS1vk!E%Q^I1_YRacM<d$A<#^XpDxAdHz#H{%Mkyd}B1urbM9W&AEMi'
            '?Be7&WWMybgWY2rvHo6!ztczQ0cQ(2eVsR_JNv_0&;5?h=A&@5{lIEz^B=Uc|KwKp-86r1iQ6Q9XBqLaUO43&5er7vz0sgeE9!S@'
            'w4+HL=4l^h$WIt_ZM?gH=CKp08umYU3%UNrXX^GtiHQ(yOly9^6Z&ttQGeKhdf0V3iLf%oA|k5Y@7aldX&#Z|5W#HmRJ=a`_}TN>'
            'yge*%{j`^ui2b<B-sWz(2vWR{O&2ivHX|SV6PPK1VJOp^XJ^Q?TW~n+Y)<Q5x;~1q@4BmaG2M3$XX0hCU!^(N21+g`xTN(_G4K>z'
            'q4p6)=wRE)Qg1jX+0?XaHwtQ*4bBIX08qGR@&nB*0p(C)YdaC}R$?kBIGpq8`Fzeu3d}>OvJ@9xv9U@Eye_3oQqUWmV-IfM0SLw+'
            'Kim~+3>X+Q_bp>w&rL}H1yx^j?ViEd1rbslD1iCxCay+7rg*A^-o#ED@4n@UVWvu4p(bw0#fcJN_~)qIp%JVQ!FpA|3cY#zUR)#f'
            'u@-`RZypN}NNL_i>Mf;}Rzc``p~Oj%nspnx)G|Uld2L{nEqhdbzIX!Drxh*gobCz!n4jCC#RRu7V&#lOaX>2-sTT>1wOs;G?VOkp'
            'KSepgG<SL}Tzf^XwW;hn%YbK#de{?HYVDprN6zsg)e*OyqtK%){Cop=ZuywvYimxpRp_<lM~koYgbbiI)1fB=<B|`%uWYP(XQENz'
            '=H`^wNv^>8mU6ZMieWsu9ACbe{_lAD@&Eqk^#A?W+0~OV-{&&<=x$~Pp5}HbhOaK2OoJ;o6aO^cowhpD1JBsWr{-hy2_}2==6X;g'
            'd;G5R?NXVKDE4u+!)u~^umJvF8-Nl2Lq9f)75co*hZz_vZ^2!4u83>Kg<YrFcq**J;PaVvN!;1dzc;_aiXJ^OzoHEkb@M9>+hgDz'
            '&*R<qeG3UK`yTh?M1XbK(3vvH5nNq=d}UxLRphi3PaWd=9m%9=`&G@}k`-TJOt4*3VmEx!6Tx_39A0zQp<`u5;2G2buM*(x5gD|6'
            'G7<ke4?|CI;$f1B4*2WE`q-O?-Wcx94tvWO3L(E)#1-em3mmn&-JZx^#@wBEJ)>_)fdR~wVGtR^0ALa(m8=yu{rKVaXB6+wLO&yw'
            'g?q%woEholQC6`#d32?cyE+q}D<&QOvYiu;^phc4P5LHlr}$d;A;5<OBp2pL>hksy{$`PWiwo!M{40*HK?{84&hy@~uqxL#^;i<@'
            '@cE>>A;?zl*<kb8`STNGPz{C|2VT!7{p$~l_}q5v=sYt|<cEnlrO07RHhVI90*4k>l*PBH3{`EJ))Jj4d`aV6A#;H#IuXwvByB6<'
            'FUS6fpoEtlG&;TLF*RkxXzTKV@v48?fzfh<!c4Uf5#u(~!*PFLr)<E|EU}?Qr4)UfN|pQa&Ka<WWvZyoPJn!>w&unS{+p)ufq)fw'
            '+hS<B((xE?96YQX1ZcPF8N)v_b*9I1L=Qf}W7dDQ$fX1=yL|zkGKZZr*nkh8ZTswrI*_)g!qp;1oC;zh32wSTTi=W>YEr1(VP)rP'
            'pndYK7zRBF@^tt*kF}(g+D@2bfM(mM=a5!0g~0-RkZn=PJOp*u#xb47t~Es`k*g|jsb=7Xt&gBzG5oOOcz<mor>9)+uw(c!KqQ|<'
            'Wb4+YW%sotaq|wmFp(1+(h*i2+X7+^9-McRgEbvIF1DVx`)auN>`)wne(Q<s+it+A3#{YeOooCv%Yo-znB9QRviqpptuLIj?0e{T'
            'b(h{EAv)wFInfHRm;k-e^Px^1|J{<k>QvTlLFVp5uv)3q4@svb55e(+1{NT%n9~!{32i<Aw2RvyNEH+v0_g7u0_i^-Tq_EDP?w&p'
            'V75k<cWs9=U+JA)U7p!C%h}aE7!W<6h=U&l<rq4{u!yT42IiPYgV2<f^dLCTngV#^6C1)Va;TiZRt)>uSk5kaC>cvBJzKnTlzbrK'
            ')VaTG=UiJ*-1xbMfwe+TWzsz6W^zn-nZ&<5n;)E58O0OZ{0t61+5?MygEb#j$_L%pvVwn<N{7`G`6Aev?uj!y69MLQ>qm5n@GLaR'
            'kblbcOM|DrMQlClw0Z2s;ku~ftVeiSXwS&=4(1bXJP}Qv*30n@C_2WYld8hoSDW$t=^Zt=7e$_a%;a9G>Kdr06S|yb2ydD0)~M<t'
            '`SLup%i*+C)@RC!3{PP<wY{S|{&{(Z8sY3}F@KiaQ*FsMRe>9fZI~pCoS2Up!phiD9lN4_+IeTm@74|RE~Wk%7gN2f(kIXeC!JfY'
            '-j!>HrSDh!1n4xhFHcXOEnuUm8?7hqM>iRGfqX&QgXZ0a9=hu6pyuI!<rFO%yrt~ljvcrmyX;hz8DyLePHg}>^X*I@2ruZBE<C00'
            ')SlCwTGR6r@d71nljc~f3pOGiOKh-c&{ihx5dh_@K@%2Gu)$1KXTE+KTL3|{XF>&?VC%A2P{o2e+nzV>>79Rvjqdxl|7f(%Vi0kT'
            'L^IO*O))JFqF7b3ypdERiVH8<MMkOMNri&p2*cM!1FM;kU^PBlg|i4MnQh2jYB5t4rbt6zuDj{MQ{x_MUC7JK@>4z0@gqPXvI>%F'
            'OKNYRb<b@}?CZ5wxyaQve)jweF3dyBg{=TK(=SP{*?U&if@j=ZsS8q3RG^I&@3>rzss11Im+^iva-FCq-yzfB-Ozx&&4p??!=k>y'
            '`TW`E6c2g!IY^V0Qv|g|X4V|g-UtYulaZ|6Z{-L6%~Z6rq<^F?;i!4rih;@wFGvf!L;yZ`S5qx>q=Eopns9M2s&;F%w_K-{DshXO'
            '#Ow{=MvzNJO-tJUD5|mK@+@ig2g|MV@xjDsEPNTo)jCb}MPKydDKz9=5w0ZOH^%E#%V0b~;kFoKew^IIfWi_jB~E~wF&4qNsW;QV'
            'jvX?3ORew?d+54q(#$xid`b!g2u6JqVZy4DOI5T?cST)FIR+Dj2Z}Cuy*jdWr=I@S=~P)Yj+MZ}tx}oWc4PgHDo<-8(%jo`sb|KF'
            'Ywr9XY$`3)Ma1r%8mcxdIq8?n9%~rlSDz$Q68u*IDkj~;Yn>vE`Of+g+Gph8Ap1}Oc7%&r_}1}o1^#rNZg5BHL?)`q?hw^jqpKrT'
            'G=72Nn$io4Oj0#uJbDJyORC4j8PHVGjVl&&RuY%y#?<2`ucXW|@2N^boWLx#O`i1TAXgwe_c1FTj6~glv49)aPIZa6QD^s=b6f|='
            '$%}ZM!s``utWr>CYmD=L#8O!{uzSPov{<FmuNKdFmr?ENVY<_)YPH|L{r=+3`{>7upI=}6a`7tq+dr@#oT@bXrIr?8Go||!OH;>K'
            'JGYF>6s*nOV#Ip0Aj1Q;DJzxEUI_o3V#4n(zPosd<<9gxUcC7?aVS2fST1!125}wlsYvTUg;u?stD{Ev4l&kt2?tWbg{A{)hSp&B'
            '`Qg%D-m$txM#srEO;mAcZ{!n`^y;zoO?i3ejB#((Wc$+dxHsdyyurgdY^aK}ag^1=D0E7-*)q+_fkt9cV<HXB4~q-*N<UR-KxH$7'
            'I?Rh(ERb)2UQ&M>ka>w0yoZTrYUcSsAA2dblu3fwGka8khAuv|`?fB$yQCJTr<fIYQ6`NFj*oeYk}-CMH#L4;F}d{TeCeQYHR^E;'
            'O|?yRvTqscu5-+~({}pIoAe`iUq`N}Xx{1^Z9No<cgOuE)2y!uW-r{Hu5NKe*E8urcvQ(tkN8^cBt>;Z0n9}(Z6xeMm>oj7hG@sz'
            'p<>v+8h`W-_fVyR<>)@-5?QuvRi<4I(?;Si<-Nb{$KS*s#knfB4zAKu4Fs&UI>m~s0yY93L!yg;MH0)QN{!Z8snS(+Mp@+Jg*dm)'
            'KDDtxm2EKB*S9mqZrW?zVD5wUoelqOq++EDp_y4}B=Ck~%Edu@0n7vmCNGwjx3pK5qz=Fyl|+RwtUWc{99nwmB&Yi`3_q5=@HS|t'
            'LD&zW%|hwzZ?4YxsTQ3mh!3FKVhF|%3jX`f4CsiigE&3)+nU988>`1BGi}@+)+@i4BNG?h&_zemQ7p2j*oP}1@FuPEV5qf`VRjGm'
            '&`Q>@c(GTvVoH*F?ld)KA~j%hu&+T}w+-+w`nHDp7|#rpV4%OpGi&NKO7;Rj$AmJv1C4zeMNRwBgJo03!=5jI`yFm^H~>?Lkr;?Z'
            'czIB<tGPjk(#Xlnr`<JX$1oV^#%*Y7clQn!=yJ8A-u3f?UI1v}ddij%!t)Qx7%BHd4fV&x)|RMU_*LKTbt(xv$HwpJ_m9n=wqtzC'
            'X+G`4wi)8U4EC3KIN6x@IG$P@P_tB2a6w~$z9$$;6{jxOaVGKJ2|9l9rY%Qm9J))VpmJJa1~>+q0t1k*6L)hOE;|ZK<g&`?D<#Fp'
            'tl5b~UZ>gk%ARoIoa6Zla^h>niP3zM0i|?83r*Y|K5_T+<M!3W<4wB;8(QL<RAvbqxIDjwZH)0-Wt6u(;;<vHgg$}Wbcw2JN}t^k'
            'wEHt5cU8J;ZS9U*t*VmKW7i<Z;Du5t4&6g6@px{fLOVEhQF6P(I;*PI-q&HgLEWXLF}pU;AQDb*(LsrMZD?pMuxLzUK|L^GYfo{1'
            '9Y{(1(WsC+7N2!TwYkr)iZwX49lHRlTMc3}p>bWzKn{y4@oFkWSnpg<$B0vt$c)lw5HY_Swe~5|9j#9+KW@ODo{t=}Pr{oqNzf0o'
            'S+^_qFr)s{<6;UU%B4vHr%`xfm*Xe=Sp2E5VnhM%<?pf#2KLVh2}ygZf9ydbcfx!~Tkxs;fSPu>l-IJC=7UiG{J_*B!*;ER>C;Z('
            'W8!$j!#Ib`y}QF5&%{ZGA~6pI*B&#5ijTEf5tUX^$6_}QAESq?z))9a_L=WtrwV5WQ;rD`SRF5tc_vG;qrfNu1pTN{R5klO-;O`M'
            'n7&N)0VvdI#c?}0vwSRdXM}q%$*~5CtTI{I3U(lx<>J&6)%xSnDd_oNEUG8kG_X`XA*Qi?7(Pg6&zU5O4+ot75H2imK;d9Tz#=bW'
            'mQY|<j~^4kdhCo%fXY3yQ(MM9ma|2cma)NLYLuNluT#2IYE|cM?*mncxPG#G>35<N$CXFZ`QC)d9i^1Phrcn={d}o=lezDU6AuH_'
            'Gmq5Q0U2BOEU<eL_&21tItcXD6skV_xCHf>iFa%jy5AN(^S+OJ#<><Lw`;MQs)IK+FXdsvl?qf{`ns5EWAmy1x!ksh=a5!&&+3hj'
            '(ig&-bfZ?2o?zUh6JGN+7S#_kVR+i8G;ZfEo#$p!zljL&yqH%u47){>`;Q9s<WWX0@2rtIpDX0=@mQXd{v08hatm=sgSVdzwJ;2Z'
            '^S8b}mA@4}7-)Yy-ulL|`c$SqwzI9@K8!p0yDh@qeBnP6U;X{uest{;k41%Nmr=nxrj;I?Rz%ssMdp9*gA^6?nNg8kZtAxgIhpGu'
            'J6xDu$<z<Gr=6;0ug~nAOHq8W9wYC47w`K6`TgI~?vLklRe<*UOzkZYONwAwlrJ&Si$2DzofieaqA%BjG~hHBeMG{!0_x-JHVS<z'
            '1TSg!JhpnZ+6C)Tw|BkmscB`xAyD;)4TE(H0^r~JqKuvM1)B|YaMk0VwDVb^p>Va&kr@2-9{DlF4!<ZQ`WOz&baoyOBGbpt0KD(9'
            '*V9$0(8qny*V@*!y2FY4spj(rQG<DGCbk{)l&|*-9Jq3dD#XUCs>mAP>SD90BsvzDZ|yGHmpv)+T}k}ZliLJ-uD&$xK<`<Jo&kc}'
            'ZMxpFswXAP6%)DQ?xjapkIqjzRJ`gNqS%np0kh@&_wY;CPDj(03+ZYU&<#_cFxO}1Io2zb7;~g=mCw8j0R2ZMw)g9V{O#+MSo*7D'
            'QCTJ3SS((SJB=t`l}-gZuR`|=cMJmT-d7jjy!h$6_t7sGumAq-d+W1uf7a#Gf5OF6eNupF#LTUlvmX9FTge~O;Tr|LhA@Ahdf4b4'
            'GLxa-v(M>frus@tH2h7SLEyS;-#2&6%M)MO8J#ZOI#^Y|Q?O{bGSTSK&Wo;CWZw%=AA)giiVTC)1>ai@sPA67pAhQ$7T<zCn^JoI'
            '1oQ8%GdWe+oYPjLbNP6aaC%4&;B11!669R2hHC@yH+4+)9sm|D+Hc>f&opn+>!wsW-xKRoOCb7`1I4tjcqJ<H=^a&GoQrp>5;T<^'
            '@|l>sJe3`c;Oz2=U3~m8p46x<znWGmS2AbwWj>FNMN>EQeJBpi;%9c~%R{sDaLiYf^IZupjn<Zh6+iy;Mt$D%#hVx3{o~#1cVCM6'
            'xwkjbu~r7H4GdAB;v`e9frCqCa0242Fr6~!6#bgG-;M}XP3=k)x4E@;?ONOG2Ll!dOlYTsQJv8aOS46pPf5tS>XVzJ`HA>db1r>F'
            '7%R-`5>7f(p0m4WocA6+Rhjys`p~c*OMyFJ{yjY0QergL#>0xYDm^tjqV3Lx<CpgLz^NN@4l){j)4VDaKy3BXYx)*3xL|^HDdB?n'
            'g7m|pxtOd{gI>ei3RK|_Y{fLouo?t?Ky>zJ+>26QQ#RsOr&n2zLch&pFGg90<<c4Bsj+@=d$bpftP=hwJKV-M<B@&6ikA@sVU!`R'
            'j!xA#f$^?|{sfy9E_~>-O%Hxsbm+69_9sOj4E6L{N{)fqN?{v2cj7b{PWc`3+YUS}v5h}c#%)h)rJ<nxX6$~8ZF<+$iuau0Wl_%k'
            'LYos^Q{Nuze8kSH0px$pZrL5SrJDK}nyYlZkD~P{-H8`@C*%~jdNj39u|J_NpVODhtx9Yu7ZBUTb)12(fu%Il^^~^dQ`lltRsW53'
            '74Oq+tmOyz0(ifIKc&jX8rP^YMdH{sR*J$ZM|W148kq`~u6Ren5xRi63$lh2)){M6+=B{AVb#yq?|=Z%3$JAqxhaYFN$a}XwHf1^'
            'IL#2<rPA3IRwD<1IbUWG+|G*DX(0*ol`PVv<jUwM>4DOK6(B_vxk2FyswvKj>qernj?#>BtY_+!sf&q@Rdl~~rb0b^o$MS^Je@lG'
            '=^rnCe61Yz9jz%Qt|vvFG$Pm)c>`7-5Uo0-wV=ZiaAs0a2;W-O26sh?>9(-F&b&>1R>g_LU=P*_2Fm<~d+>?+dd^N(dx-|lkv>A7'
            '9$`+$_yCBm<K35>Qkg;Tog!`*I#cogoZ%gDAm0(`ht&&nPE<c4pxIgPZfDg?!a*aG)`$g%+ndKrM23XU%Yz?;TDq3(?||>slg7fa'
            'Xf#Cb_h_n1{2a&?9~u)@q}?vBx|eRL3KE+i6XE>9H`Hi}`fPIRa-P-gUact+FKNaDG^b6fC9+`%E60>rsZ1Y9dKQ~Xx}N_2PS}37'
            'WuiX)()PYopY^z!I1lmL9#^B@Di==qzm9e?ALHP!ufUp^dRz3QEwvTu>c+N#)!KW&mQsB}0Yo{lOu7a~rh0lluZ+mk71|Jl0!G)A'
            'fIVI+lS`XS^24FPt=M|sd|`2nwpxszIH0;Gq)3cahT=-sb)-_s<k;wl0@wl#Y+sPvMlO*tBHo&>7sTS(T+96S@+f^A8~wKBa}tkW'
            'vSS7V)hGpUYr1L079N^uFl$}2*(8c#K2g%TSCy(TbUvYjcLJ^?qVdDsZ`p96RV_1z9X2jU)8T=y`-)7O(vVyVx7+J!FYZpDzISq8'
            '+o@Ns&x+Y{r^{&4w_19^PR04DyT#5RNL{-$=ZW%`pNMa82OxtU3UthqGTi|0)a*2283j`dw5@wzZHq>w!cnlty<DuY7aM@IbxIro'
            'Gj6~Vrab^0r@m6d1_#HuEkIu7R>R!L8ep6?I7GZIi%L7le2tg+dSJb|u+?+D?=8aH1>rqX+C8kgrfrWwwpG2IdZ^-45Ga2Q`wByK'
            '%T9Mb!FsyT^rWxO&rq0{cQyUr4#g~O&S}wJ_C@zk)@oYzlf|}Hl?Bv$*4gu6>M?klc+Jf!$_v8aXdk@`R_MSDAe7T82Bme3(XIij'
            '9B&)ZoszI$@2RLqMXtr2Ul60c)~>!y&W6wL@W~#mmoiu?<$;PQZAA=FCV}UAS_Lx91jEObSkcJ)M`b6H82T<t_h&FHALYJTrp@nS'
            'QGMF9^n%ZpTjjXF%ZHO5me%W0zX!a0&>>!)ID_c~)R+Fbm=>*ceW9x)Iu|c6(|}t0%<Q++UnmI1fcv$qr(|zTwW`JM5}a1OL;OSr'
            ')OeGAkja#rPK{VVsNvA@n0IvL)YIyyvz}tK&e^5&4P(3TbElhgpy`&%HG;pVn^rzOq{eC@rx)wq7K!cAOP^mzr!RdkhWnbY`FdT^'
            'cIFH$(;|Erit};a2O&DGe}~iCqTrsLUx_D6G5%8g^5VxguiyOLKEM{Ycz>Eg2mv0K>5BF>rJRN1fwL*+Hb_sluUB_s_!fs%GF}xN'
            'fX#wtHUrXEGy+C3Rimov?C&rrOdsEj^P?TE;IKrXrr1U;LJm>j_cJTskl>I!n!@A;lyU<OgP=-R#aeP0B?cN~c&UlZv~2h$iB(=S'
            '(0LKgH4gi<(M>MIZYo2SsCe?81Y%l&b~<8Q9NY}L6|oR!SgiH_%uN$UAKu(Ta>gIH{9^PFE)gYv;wH5>=zK8S@5~zDjDjv?QBWGH'
            'lf2{F<T?*Tt6s`kBD!*7=|ns}dy6g%PRt!a=U8*>e(*j`KJ*va(vw|u3TSrjytpWLv$y=4Fx-66s{#CFwupIo2gu%EbuYx=`lrsd'
            '7&npf{{t)>gmM'
        ),
    },
    'approval': {
        'sha256': '8e6a6e6d92d0b51fe6ce55d06c96ec76680530d63e2335bf2d6961293a8f8357',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;NTXWmSl78o}m;*lutfF{ocPpt=bm>$n5}k0PNM1=!)~j-nATT5mf&dp6%d|@WeZTIW0R|u`Ih)=1!IA(>qo*%l-v*xNT?A>C'
            '#$k}CZB~@0OP%UGDB~<ut7;RZr@?lcXP<F<8I*e1>;3lAM|J<x<WBu^bNS(Ftj7O2xx2sXogVzxJGZMhRZ*(BR)M-3zdye{zaOij'
            ')LT`qf>N#U5}g+6{N4Td7MHZD3Z3I}p@Oiif+X3gLhH0gw+x3mQbAtE3p_AX?}MU<)1{gfVIFVGsW7dhv-`Z#GZm+0#$*>&8uH%g'
            'wLMFz;Cr%EZFG>z1xsGV1$YfMTfEY<lsyoYWx+-(h%f2!R>E8}Z*i$kPqE=emV;8d6KSe2tC9#C`>Yinq{oKpp4Q<VX;$J%Wd+_N'
            'fe(lf?gc8Z=5xq|8G#z^WJ8X+bXw*+C=r`2h&d~Z=8@rcHyfx~Sr+sg-WBy`xlSXUPpdLc<Z7goc+O&h(1S44kjeE;%@w4wtTuQT'
            'Eyuo7m8E!4f7W?VZ@?@levb9yG}r%K#kr1V%Fl8&TLf`JfBap|;<Tt1i#UuyWf}!#fUCV(1QJQMCDPMv5`=n{CE#q<hj#O9lZo0g'
            'HM1I+)n=o=gh^GzpNZp@&h?;2yfI&oD=lgw4Uzb&?KX*x#_xVQKmFJ5zE|0T|I}Yd*<qHJv^{J^cA_JQ`-jQ(<>dNDb^iYS?ai;}'
            'SL*&oUEEyXkFW2Kn7-Km7;@P)!pn7-<x$aWVrpJDEmi?ofqICOC|+u~i{Z<%(VOqcR()frrT$VvoIw@EnOZ=)_~UaNY54<Un}sVK'
            'uBZ0pet!m1r;sTmRfx5k7cC&*mdT$KDy;IH<X{*as3OC1YfSL9z<A7{2da6MrgZWC{%)Y4Z@R>mMHh*(a28Z$w!!j{T{=#|RvgtH'
            'xTT$DRne>E1yd=>O(2N^n@UTp3WGe~5m{&m-t`#dq@e4pT+v0+zpA!L7DUBBZL4__7b{&bQ?wOqj=U+=<ztqwfhzguI0UYFp4VeB'
            'Os9*gta3e_s(7=_ax!#yCOc5k>zQkSrWO3e{`q^6rLvrMQ^tS<bI1M-#p3_3S*qo^ZBP;c``CN@+3OkW`r&Fk{TT*#r_Pk`K@_28'
            'cq6|^y@B#X@buf~4BW$za{5Cg?)7`UHvkoqr(LONrQpcr_+oN5xw)=&JrLjl@I`S@Umd}90=mq&1%hIQ1rcEr)nQzaOU7lPlZEm('
            'S)|EGsyxKcO-7DfusLVs0qHWq<*X{VRS9aqL9!{~m=v>!mwmP|(5rorq|7f2)^KCmrkc@C9z6Sc@6&7ua^hx-SOZ)Ih8btW72d05'
            'Cv0Hx7U-6O`wS+>At`|~2$bWQd6I=|;5H}%vc;glask^lI2>t_J*^^;VL~Lo#+F?SANHWoi}BqZuw(lE>il|Y#XtRd{1NJHHbP2U'
            '!hJ+bi(Eroc{T^`h04K3H4h5j17I;r;&3-2&IpH7!eB%kV&0$&89jcBDh_&6ia0w`w}KCrc-W}eVB;1+R8!~}_O8djj&G-qInpna'
            '%keFl<de5V2=ShY<8!ORO|TQhr5JBt&*)1E1x!wE$MDOd1QQ!lOf2?j9wjA8t{~7Na51mq#Gn&4Sfw>#1A(N0z?IIa6FES!_73&P'
            '_K*Y>Gh42*Lc8?h6lPgSo`5$wQNH#Pkb7Lk;Y!W1gLNlEAcM_06A?>HDJftmvUv}%tswQCUtcOpO7JdrDvrMJuy_F=e(<c}c%L3T'
            '5RGDn9q-dy?DLli+mSJI_8<tbQ161I0D|3)|KkIMd^x=xUqe$<gs02NJxq;Zq^I!DD|NWyWN&J$mCS<m9(e<Kq?19}%e5$9xI^xW'
            'y_ASM#nfU{4{&sIJNa>PeSS4P|8P0E`I8I^v`pazTFlyE2=D4edVzE#NGkZ^)CibTDZhcXC=ScMI{hdAG7_?$H#Un3&SrL`vRVZc'
            'X^gRofeK*+9$I{>5iyX6Je^eOMwcfe^*|O-d>W{eMvJ)oG#n0}aZ$3NY3tqt?tOY5GQo4CbDM3sQ-&a5swge=-dtaOq&%=>;1Z(@'
            'Tt9SN&@lJ|qNBlPm<-I)f)$y5q_?z|f-52>yT<|37BMMxYLgeBTf_g%+&on}06RAMQ;Q^6);_=q79fJ`-%xw)pTCJEfSvtQ?0jHw'
            'u5r=+0mk;^J0sQ(`67$Cp&$Wj*cjhXjE~ooTsU6TR*H=lRl<Snl(Q{tncV&xcZW=B#qtK`lsUNLfI}!Mc^@T<Mu^r(`6nC2h>hX_'
            'i24cs^YBSD-EdA$RaZz$UHYDE&g)ZHczW*3#b$xM!F1w<LrR`BZrHcA3aN2a@058Oaz-BcFZyv`IJL@}O4L15ZDwmEbh&C2O3$|l'
            '$SMaj-E}tbPbo{;7mVZ|9l^BLP<oXj@g$c4sD#1R*qycXX^S<#7RzhUS3d}Wceq4$kmd0*PO(MK@~1^LFXJ-Ng;y`r@CkJs`uREG'
            'ln{*L5<ues*CUsc<}%BImbKK*k(Y@{?qe*nW%ezDU<#u5J>NW{%Dv}|#K9iFw4mZ3NJH)KJvm_4=r{blQ5T%WOQxA`j4KPK3eqAP'
            'P;OFEL}k`3(gbdn;>c*9!Knc&?^~tv0twkGX}4MLYH)95`!yt_m(}x<igJ!;>ZxAf@sr-jNT#{&Xh*`T-(2ZC&+RL7w7&)C`2z;('
            '3E=q$Zn<x2yX~m$RPx-j$r7aIo>C$0(|;pG)tvJJ{H)W0cU6)oAITPK+>rdgUpEd``ioPm2eb^jL-G9K{@v+66xZuR3))7grg9w0'
            '%YA+^;g09^hdK?xr0-Sb;`AS0zdytagudTz^Bmf)-IPjCKb!x)$G6knpf1h)cg=PE#rgHk_2lCGO8xsCsz81vg&n^@9g1MH*1Ljh'
            'gkp<Sy6Bq>qG=NrnG{P1d8nRa!24HQ<Z~~qt_*|ZG1wJqN9k%&-)Cw@Xt`dZdIz2r#3kOWDB$HJ4x}{@9<e}?H={WD7#Gw(<We@*'
            'RN?DL5QPixxLBLAj|@71M5X+rBtDH=1sdcOC&?*aaB4L-#EL2+U=s;=7*(5X(V#k{Ilv<Y^$CI^jN>ztoRAQwBtq^y;%~hT5Hzy9'
            'IP<*$MUjz*Iv1J@kT;w`Ut*Gbx_`lB3~Mh~2Q`3v>M;BnV<sZ7m$ng`2hO@Y3BQjq(##BK!Y`+e#2i5t018Dw&21zLS5>+uaDt+9'
            'KM6MTC>WV1hg7Qi-+%YrzkLsB&EXqu@DetL)t2<>Go_Zq%>8`<*>)S3D&WFjBmZT;AE*QMfh#(j<R9o9+J1%UmT*w{vW36FfXd9Y'
            'PH59it6WjxYx8Snz-+G5>fx9K@^#BbGNqSWEd$$4F*Hr7T>mpP-6o>6aLn^f&D?t4>AG{n@!jNVEVg0Rnfjyqo&}{Zrkr=yNZAy`'
            ')NuF}WtSf3IeWcn*EZz`lCGG0EJlpMJZS{J)1~tDa4EhUkN^?PPdSA)I|PKRgQX@jLfTdVLP{>oC7a<E4;B3q;oBI1AWGV?LA2YQ'
            'YfLhyjX3-87v4NZ(Anr6_Jh5^-PaIyl~<?%iRRz(XXVfJI)PYIT<g+)?%OokGC}EJw-wlMP=jtKjee>9)OWL)PD*9z4U8{P=7A^d'
            '<+eBOZGSJ(=yE<JP*rhA#iu&4R8PpjmzCOO>$fO_f*qa=!eFiDG0=RUXj~*30cQ<_&0_^D2C8o~GN@tu;k}L87fPbZi9-L+ZppBb'
            'Mu9pZiYLJS(C|aML8yect*L(pm*ATST6!5LQJf!T*x=dWsBgD%z*0>2rP-8e>#)CfOD*S~w<jy-<{uW3^k|T(36Yl8U521=<R6IV'
            'p*AiIw_QV>{kMno-Zc#XRvxdv$pG5&<jl`u<bWWX279x6DaI(o?N_b!t$j&4%Zxiw*WYh0_lI#IvHLY^TtKuie&oM7Wn5XYgH@?{'
            'l~l;$gD3`~G-QD_-Z2UA+Z+x|6M^8`Ztz>02#oU6W?8#bbI7?HZ!RC)_<>EE7;|vH4Yg*!2lnc`)T9K`(w}Qe2c{BZUdaktT|=7|'
            ')I$ayU8h<{JcX5nwoU>DxE3Z=WHV-Ni7_djb>PxY+EhARWxxuAdnY=nYpzH)Rawu>YUR{HD_vyNq;D!B+6)RI`r*_`cHxvdi4|2>'
            '9Z-<V{dTeE%qp?s8nD(X+H8QvJxPe0o1D9l8+BLl4MNdYsrT^7)@qzG<vXW8fgL_{nV}xo!Q`&dV827bVV73dT}l0ghu%(F67Y<<'
            '%YoeyErzN1cx2Db=*gErMMfAc!E=w9=Xhj4$0sqPm$$?6e}UsRTmw*oWfJTF&g$f;#n6dm=%oLAqPAHA9jyZ-gs9$Ph$ac@SiK3p'
            'cJSKskIeP^4n+6&1Y{<eZUg7O9wJkVn+rJV<sz&3IKJe9!lc=L+cKzAjw-a!GQtiVlBPJ*%A2T19$V0X49_%Pd8mdrtjQQY$AP(Q'
            'M+)*b$35NvkNDUC`!Atf(`u%)*|w25fkk<n=w`@R560rcq1Ozkn#L1NGN8Q|2OO=)Lw6ob|K?MtN%XqibZn>Nru01bWLhujA6tFU'
            '^4Ln=QHxaFT%Uhb<Le*CS636N`q?B?sGCojK<n$QK>dm=YjhwjSa?q-19dX|J&RLe$P7<D^`D38Rze4nfzhU%I#j5gkgf$WrRJMM'
            'nFbQJ6q&_mt#%diyZAI%qi(3#P$f)$10Ex4(}EOS8hWMN|8PpzLIsXKM#EF9yUvtqpqx--4}x?KjmX_2BZvB+$$V$1z%q5gztjBL'
            '+mC2++(y$W6~oJnf{+A+vlC14gbz5u<ar(yOZ_-I!sAStx!^0uOl;-eo~s3b!QYrk<<oi|*V=xxM8l{*JWx+k7b@Ku(J!hZNCX5-'
            '@3A?U;||aj5=c35z(YS(R3Rn$iz=Zs-mpe+#38#r4+VBm`$yF5*-6JqHv0&`({e6r<Q_$0n4ptjOfShY8XlP(!a$kiTFRFh-!|u?'
            'UCjZZKB*;}mpn@^hw7Ykln6a=zN|FOTk2C3x&EEAakGPWc!Z4W>_jl+gef1M)HAO^B!16lcbpj`Q}YY`(i{TX`S$UC&*@vR298Yl'
            'on8fR4+^F;4^^4R%T*~8yFjw5_Wb`3P5%GNo<KWqc769aUC((;ea-SJO+ZnAZXIvHBm_hNg$i{SQQ!_YI*EtvcOAo)-nW%|-jVf|'
            'p`p4-Lvq|ubIrc0LPW+mjmaC}ZZVYIO&yk5Q8J#Ub-xYuG94+S)IUd9P&*_Xj6wq`0<lRv5b8Bp*e91q)v`*flm!vD%WhNoyipfs'
            'jQhj*-3=U}4$(YllQe^Ds2E5iTw7q_c}vkr>fgw~#GR)xO4TRT5)A+-99TGUT7wmgvZAbH_<Dl$f0-A2LuJpQ8lHYRzMNcLPDZx<'
            'CuT<_J|~FlB(9@n9nPbebap6!y8+)WgLAZkIb!*NKyAXi1a@w`#La0hM0r;G;b?xe!EhkH+eB9){Gp|H(#43sBD~Nd@j6{@^jeXf'
            '@3Oq$`Ne)N>G#!|ww&&z4l?_eNXgaBHxxT$e7OWfadEcdCfTzlb;NOpG|Srh{b7<l>Kv^7?X1Otz5cBLvDh~m0Rxt;nsXbBecg0('
            '=p`qy@}G_{_%QmH?_mtXVZWn>g=fJPMtJw({QBpK1&191vH#?5!=XYAm108*8mfu#ARI>VAvh_rGzS3b6rn7&y(gGcmWXw{Jp^{J'
            'z#N`nF^^3V6zpdA)ZhPquO0ue*8WVL*b$*yHXf@JNuD`;dJU0vipHLs-@+q!{M^Pmd-uV%GpPP(V*J5p=$K?dCrk0@!jpkc{gz-s'
            '=O#^hhIsqr1mEW_JzhI)j`X$LGBp<?Zaob4?|>$?WtzVZJrmGa-kvfZrrD!!4;l}vGNjI026RC7?YpfUE&{NSe^QdwY04ZqpN<u-'
            '_qZ(@Q`*!yoSC<^kDwndZVvU;>{0c-lA%>9$ae_Y2BkbfU8&5c9&*>PW)4yXWs~u<xVp#p>e>c{Bu|Iw7qN$vXX`Ze$0b}UxA7ml'
            'FsDYW#rV#YshL6dyR>9n%(wY>p;dEh`W+CLNfz2=(@WG0m!M_^XXfInAcU&l@s<dG+rhQwe14(s?$58rtOU33!Kc~C3XWGAb&1gw'
            'Va<^sI-_`gb$3Ij3~WvlQOYGQ)|{|b%Z$$~%PgPghcpj#XPhp#Nnre!tF6;hY50BSsbf|xSI)Nz^0fv2-V{0^GZe)d;kBy*IR?mQ'
            'qaC+4X9DDio%6Q=m=TuOhwTnGUn%N36rJuOz3;(mK~-H-Fs{ZjTtZ<cb$bV_<?2ZvMMvN3b}fZ`ujjt1=AzXrtzBzxPS63r_PpR2'
            'ymU$uq5a~vJ7vgzlLFC29eJJ90Wxl!SJGnYgeXZIw3?gD6cK(v6%-19_$y-J4naSlVybs^*5nGa)yK9z{Edz4*W1zut)i%cbRBoH'
            's+BH+t%P14o`B5ri&j8q0)<^SA+#DgkPN=2DU(_JIKDtSGWoBOWssB*!M`QFA`>n)drA7j^KI^so;_JE>Rb*u!J9s_sa})i4b?}J'
            'v6<`>Wv3zaNA&nc{pw&um0Cmvr`I?4s`<Xnw2%b%Ms3{V)`}s=*jYEVn{)%P+8j1^&h|Rb)2Z6Ox<GtQyG@PQC|FQcG^!<2E}-GS'
            'EghZKD)F>_sY;|wz1t-|UNjyX2)}}Q0^J?R+uJ;6vdZ-|0~s-8_zV|G+&Tql-^EGN!D6tRD!thJZSLAO?+bza)dujb`AV^Lz5r9d'
            'o!?*lG`^J6OTy|v{WiJ(>E^?|^JPR_)(uiV1%*1Nu$>ZaY8av>g_ch^#d8SrlMHrUp8~S3BK5ERW=PE58IGt4*5q_vkh-;Ys55KH'
            'b7l@t(MPXxV8;2(mA~R5$K`&Dnn@IyQOKIL;~0Mj5Y*Emd4+Ts`I{n1IsV=>CMl6|KV0Hr)ZRC;NBiAxZ4K|he%$^nRQkL1hiKeq'
            'uTwd3Is=0VeWJw&CHH!otrp<R?MCOZrrl_3-MZ(v3%2Fap5tq+42xdf5cqPhQq?oSzo45jxxN@*jei-dpU*#DPFfYR)63uu`cE_+'
            'KEhf5g+7J>6w%?)z6A{uo!-S>g<RXo!3x>leehcD(k)i)WBbLbV-DWPwV^}L{O^j@BceN4jJ{UgKAV6mE>3-athYmrPQ$gvYV-ZI'
            'pfw#lo&rXuT-%zHuI)m$thR}M5ZtE!Kh@BBk)<d$kQsz}#FWU{`5<=P^=(cwQ}h=eWGd`8pX_$(YBGerOtzxzF*OOs7Glvn+LGA_'
            'm&WpqgN#^cTI77$Ur2^gA0^p8&g%{81LJ4*r1GKjz4d0y@oCzrFAuc_p+`5=+#t#Jd;bGEZ&}L'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
render_utils = _local_pipeline_modules['render_utils']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']



In [ ]:
ffmpeg_bin, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
print('FFmpeg:', ffmpeg_bin)
print('System ffprobe:', ffprobe_bin or 'bulunamadi (probe_media() ffmpeg fallback kullanacak)')


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None


## 1. Girdi dosyalarını yükle: `hot_moments.json`, `selected_script.json`, `selected_voiceover.json`

In [ ]:
def load_json_artifact(filename, override_path=''):
    candidates = [
        override_path,
        str(RESULTS_DIR / filename),
        f'results/{filename}',
        f'../results/{filename}',
        f'/content/results/{filename}',
    ]
    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f'{filename} bulunamadi. Aranan yollar: ' + ', '.join(c for c in candidates if c)
        )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f), path

hot_moments_data, hot_moments_path = load_json_artifact('hot_moments.json')
selected_script, selected_script_path = load_json_artifact('selected_script.json')
selected_voiceover, selected_voiceover_path = load_json_artifact('selected_voiceover.json')

# Kanonik sema dogrulamasi (Notebook 01/02 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors))

print('hot_moments.json      :', hot_moments_path, '| status:', hot_moments_data.get('status'))
print('selected_script.json  :', selected_script_path, '| status:', selected_script.get('status'))
print('selected_voiceover.json:', selected_voiceover_path, '| status:', selected_voiceover.get('status'))

if selected_script.get('status') != 'approved':
    print(f"UYARI: selected_script.json status='{selected_script.get('status')}' (approved degil)")
if selected_voiceover.get('status') != 'approved':
    print(f"UYARI: selected_voiceover.json status='{selected_voiceover.get('status')}' (approved degil)")
print('Not: onaylanmamis girdilerle render yine calisir (inceleme icin) ama nihai durum')
print('     review_required/failed kalir, ASLA approved/production_ready olmaz.')


## 2. Render yapılandırması ve onay bayrağı

Tüm render mantığı tek bir kanonik çağrıda (`render_utils.run_render_pipeline`) toplanmıştır: aralık kesme, altyazı hizalama (gerçek seslendirme sesinden `faster-whisper` ile; başarısız olursa orantılı fallback), dikey kadraj + altyazı yakma + ses miksleme (tek `ffmpeg` geçişi), teknik doğrulama ve onay kapısı değerlendirmesi.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' veya 'center_crop'
TARGET_WIDTH = 1080
TARGET_HEIGHT = 1920
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi

DUCKING_ENABLED = True
ORIGINAL_AUDIO_BASE_VOLUME = 0.35
NARRATION_VOLUME = 1.0
DUCK_RATIO = 8

SUBTITLE_FONTSIZE = 20
SUBTITLE_MARGIN_V = 90
SUBTITLE_MAX_CHARS_PER_CUE = 42

SOURCE_VIDEO_PATH_OVERRIDE = ''  # zorla belirli bir ham kaynak video vermek istersen buraya yaz
print('Render modu:', RENDER_MODE, '| onizleme mi:', PREVIEW_MODE)
print("NOT: onay bu hucrede DEGIL, bolum 5'te (videoyu izledikten sonra) verilir.")


## 3. Render çalıştır (kanonik `render_utils.run_render_pipeline`)

In [ ]:
render_manifest = render_utils.run_render_pipeline(
    hot_moments_data, selected_script, selected_voiceover,
    hot_moments_path, selected_script_path, selected_voiceover_path,
    render_dir=RENDER_DIR,
    render_mode=RENDER_MODE, target_width=TARGET_WIDTH, target_height=TARGET_HEIGHT,
    preview_mode=PREVIEW_MODE, ducking_enabled=DUCKING_ENABLED,
    original_audio_base_volume=ORIGINAL_AUDIO_BASE_VOLUME, narration_volume=NARRATION_VOLUME,
    duck_ratio=DUCK_RATIO, subtitle_fontsize=SUBTITLE_FONTSIZE,
    subtitle_margin_v=SUBTITLE_MARGIN_V, subtitle_max_chars_per_cue=SUBTITLE_MAX_CHARS_PER_CUE,
    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
)

print('Kaynak cozumlemesi:', render_manifest['inputs']['source_description'])
print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
      f"({render_manifest['subtitles']['cue_count']} satir)")
print('Render dosyasi:', render_manifest['output']['final_video_path'],
      f"({render_manifest['output']['render_latency_sec']}s)")
print()
print('Teknik dogrulama (passed):', render_manifest['output']['validation']['passed'],
      '| probe backend:', render_manifest['output']['validation']['details']['probe_backend'])
print(json.dumps(render_manifest['output']['validation']['checks'], indent=2))
print()
print('Onay oncesi kosullar:', json.dumps(render_manifest['approval']['conditions'], indent=2))
if render_manifest['approval']['failed_conditions']:
    print('Saglanmayan kosul(lar):', render_manifest['approval']['failed_conditions'])
print('Render durumu:', render_manifest['status'], '-', render_manifest['status_reason'])


## 5. Render'ı izle ve ONAYLA (ayrı adım)

Videoyu **izledikten sonra** aşağıda yazdırılan SHA-256'yı `RENDER_REVIEWED_SHA256` alanına yapıştır.

Render onayı yalnızca **tüm** koşullar sağlandığında verilebilir: teknik doğrulama geçmiş **ve**
script onaylı **ve** seslendirme onaylı **ve** seslendirme-script hash eşleşmesi sağlanmış olmalı.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır.

Sadece izlediğini kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap — bu `status`'ü değiştirmez.

In [ ]:
video_path = render_manifest['output']['final_video_path']
try:
    from IPython.display import Video, display as _display
    _display(Video(video_path, embed=True, width=320))
except Exception as exc:
    print('Video oynatici gosterilemedi:', exc)
    print('Dosyayi elle izle:', video_path)

render_review_info = approval.describe_for_review('render', render_manifest)
print('Icerik SHA-256:', render_review_info['content_sha256'])
if not render_review_info['reviewable']:
    print()
    print('!!! FINAL ONAY ENGELLI !!!')
    print(render_review_info['blocker'])
    for condition in render_review_info.get('blocking_conditions', []):
        print('  -', condition)

RENDER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''
RECORD_RENDER_REVIEW_ONLY = False  # True: sadece "izledim" kaydi, status DEGISMEZ

if not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Render ONAYLANDI (status:', render_manifest['status'], ')')
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## 6. Dışa aktarım: `render_manifest.json`

In [ ]:
render_manifest_path = RENDER_DIR / 'render_manifest.json'
with open(render_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_manifest_path, '(status:', render_manifest['status'], ')')

render_validation_path = RENDER_DIR / 'render_validation.json'
with open(render_validation_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest['output']['validation'], f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_validation_path)

print('HATIRLATMA: bu notebook hicbir yere yukleme yapmadi (not_uploaded='
      f"{render_manifest['not_uploaded']}).")
print('Yayin, ayri ve elle onayli bir adimdir.')
